# Monte Carlo Simulation for External Beam Radiation Therapy (EBRT) with Photons — Varian TrueBeam SVC 3.0 with HD 120 MLC

---

## What is this notebook and why does it exist?

This notebook implements a **complete Monte Carlo simulation pipeline** for External Beam Radiation Therapy (EBRT) with megavoltage photon beams produced by a **Varian TrueBeam SVC 3.0 linear accelerator** equipped with the **HD 120 MLC** (High-Definition Multi-Leaf Collimator). The ultimate goal is to generate thousands of three-dimensional absorbed dose distribution maps that will serve as training data for a deep neural network capable of predicting dose in real time.

In routine clinical practice, computing the dose that each point in a patient's body will receive during radiation therapy is a computationally expensive process. The most accurate algorithms, such as Monte Carlo simulations, can take hours to complete a single treatment plan. The AcurosXB algorithm in the Eclipse treatment planning system (TPS), which is the standard at our reference hospital, typically takes 3–4 minutes per plan. Even this time becomes a bottleneck when the inverse planning optimizer needs to recalculate the dose dozens of times during optimization. If we train a neural network with enough examples of high-quality MC simulations, we can obtain predictions with comparable accuracy in a matter of seconds. This approach is known as **deep learning dose prediction** and is an active area of research in medical physics.

## What is External Beam Radiation Therapy (EBRT)?

Radiation therapy is one of the three major cancer treatment modalities (along with surgery and chemotherapy). In **external beam radiation therapy**, a linear particle accelerator (commonly called a *linac*) generates an ionizing radiation beam directed at the patient's tumor from outside the body. The radiation damages the DNA of tumor cells, preventing them from reproducing. The fundamental challenge of radiation therapy is to deposit the highest possible dose in the tumor (to destroy it) while minimizing the dose to surrounding healthy tissues (to avoid side effects).

In the modality we simulate here, the linac produces **megavoltage photons**. The Varian TrueBeam SVC 3.0 offers 5 photon beam qualities: **6 MV, 10 MV, and 15 MV with flattening filter (FF)**, and **6 MV FFF and 10 MV FFF without flattening filter (FFF, *Flattening Filter Free*)**. The photons are very high-energy X-rays produced through the *bremsstrahlung* (braking radiation) process. The linac does not produce photons of a single energy, but rather a **continuous spectrum** ranging from nearly zero up to the maximum energy of the electron that strikes the tungsten target.

## Key differences between this notebook (EBRT) and the previous one (IORT)

In previous work, we developed a similar notebook for **intraoperative radiation therapy (IORT)** with electrons. The differences between IORT and EBRT are fundamental and affect every aspect of the simulation:

| Aspect | IORT (electrons) | EBRT (TrueBeam SVC 3.0) |
|--------|-----------------|-------------------------|
| **Particle** | Direct electrons | Photons (bremsstrahlung) |
| **Source** | Monoenergetic (6-12 MeV) | Continuous spectrum (0–15 MeV) |
| **Filter** | N/A | FF or FFF (flattening filter free) |
| **Range** | Finite (R80 ~ 2–4 cm) | Exponential attenuation (infinite) |
| **Surface dose** | ~90–100% | 31–56% (TrueBeam commissioning) |
| **Collimation** | Cone applicator | Jaws + HD 120 MLC (2.5/5.0 mm) |
| **Treatment** | Intraoperative, single incidence | Fractionated, multiple fields |
| **MC speed** | ~2000 part/s/core | ~150 part/s/core (with MLC) |
| **Patient type** | N/A | Pelvis, head & neck, lung |
| **Clinical data** | N/A | DICOM from Eclipse v17.0.0 |

These fundamental differences require a complete rewrite of the simulation. The bremsstrahlung spectra, the multi-level collimation chain (jaws + MLC), the extended interaction physics (Compton scattering dominates over electron interactions in this energy range), and the clinical parametric diversity (3 patient types × 5 beam qualities × continuous geometric variables) all require new code.

## Notebook structure

1. **Environment setup**: Hardware diagnostics, package installation
2. **Photon-matter interaction physics**: Photoelectric, Compton, pair production
3. **Bremsstrahlung spectra**: TrueBeam SVC 3.0 commissioning-calibrated spectra
4. **Simulation architecture**: Source model, jaws, HD 120 MLC, patient phantoms
5. **Main simulation function**: `run_ebrt_simplified()` — complete TrueBeam head model
6. **Clinical scenario generator**: Stratified sampling with patient-beam correlations
7. **Batch execution**: Multiprocessing with timeout and automatic resume
8. **Dose analysis**: MHD loading, visualization, DVH computation
9. **HDF5 dataset**: Compilation of MC results for neural network training

## References

- Agostinelli, S. et al. (2003). *GEANT4 — a simulation toolkit*. NIMA 506(3), 250–303.
- Sarrut, D. et al. (2022). *The OpenGATE ecosystem for Monte Carlo simulation in medical physics*. PMB 67(18), 184001.
- Saravanakumar, A. et al. (2025). *Commissioning and beam data validation of Varian TrueBeam*. Reports of Practical Oncology and Radiotherapy.
- Mohan, R. et al. (1985). *Energy and angular distributions of photons from medical linear accelerators*. Med Phys 12(5), 592–597.
- Fogliata, A. et al. (2012). *Definition of parameters for quality assurance of flattening filter free (FFF) photon beams*. Med Phys 39(10), 6455–6464.
- ICRU Report 83 (2010). *Prescribing, Recording, and Reporting Photon-Beam IMRT*.


## 1.1 — Available compute resources (Kaggle)

This notebook is designed to run on **Kaggle**, a Google data science platform that offers free GPUs and CPUs with certain limitations. Understanding these limits is crucial for correctly sizing the simulations.

| Resource | Specification | How we use it |
|----------|--------------|---------------|
| **CPU** | 4 cores Intel Xeon (≤ 2.2 GHz) | All MC simulation. GEANT4 is CPU-only |
| **RAM** | 30 GB | Store 3D dose arrays (each can occupy ~120 MB) |
| **GPU** | NVIDIA P100 or T4 (16 GB VRAM) | **Not used** — GEANT4 does not support GPU |
| **Disk** | 20 GB (persistent output) | ~15 MB per simulation × 350 sims = ~5 GB |
| **Maximum runtime** | 12 hours | ~10 h effective MC simulation |

### Why doesn't GEANT4 use GPU?

GEANT4 was designed in the 1990s for high-energy physics simulations at CERN. Its architecture is based on sequential tracking of each particle through a hierarchy of geometric volumes. Each step depends on the result of the previous step (which interaction occurred, what secondary particles were produced, in which direction they were ejected). This inherently **sequential and branching** nature makes GPU parallelization very difficult, as GPUs are optimized for SIMD (Same Instruction Multiple Data) operations on homogeneous data.

Experimental projects like *Opticks* and *Celeritas* attempt to port parts of MC transport to GPU, but they are not integrated into OpenGATE and are not mature for clinical use.

### Impact of using photons instead of electrons

Monte Carlo simulation efficiency depends heavily on the particle type. **Photons are ~5–10× slower** than electrons for the following reasons:

1. **Photons interact less frequently**: A photon can travel several centimeters without interacting. When it finally interacts (e.g., Compton scattering), it produces a secondary electron that in turn generates a cascade of interactions — all consuming computation time.
2. **More secondary particles**: Each primary photon can generate dozens of secondary electrons, each with its own trajectory that GEANT4 must follow step by step.
3. **Larger field**: In IORT we used fields of ~5 cm diameter; in EBRT, fields reach 25×25 cm². The phantom is also much larger (40 cm vs 15 cm), requiring more volume for particle tracking.


In [ ]:
# =============================================================================
# COMPLETE EXECUTION ENVIRONMENT DIAGNOSTICS
# =============================================================================
# This cell verifies available hardware and configures input/output paths.
# This is especially important on Kaggle, where the environment may vary
# between executions (you may be assigned different machines).
#
# Why does hardware matter?
# - The number of CPU cores determines how many GEANT4 threads we can run
#   in parallel. More threads = faster simulation (near-linear scaling).
# - Available RAM limits the size of 3D arrays we can handle.
#   A 200x200x200 float32 dose map occupies ~30 MB.
# - Free disk determines how many simulations we can save before
#   running out of space.
# =============================================================================

import platform         # To detect OS and architecture
import os               # File system operations
import shutil           # To query disk space
import multiprocessing  # To count available CPU cores

print("=" * 65)
print("  ENVIRONMENT DIAGNOSTICS — EBRT Monte Carlo")
print("=" * 65)

# --- Operating system information ---
# We need to know if we're on Linux (Kaggle uses Debian/Ubuntu)
# or another system, because some paths and commands differ.
print(f"\n  SYSTEM:")
print(f"  OS:        {platform.system()} {platform.release()}")
print(f"  Python:    {platform.python_version()}")
print(f"  Arch:      {platform.machine()}")

# --- CPU information ---
# GEANT4 uses one thread per core. On Kaggle we have 4 Xeon cores.
# Each core runs a fraction of the N total particles and at the end
# results are combined (statistical reduction of the dose map).
cpu_count = multiprocessing.cpu_count()
print(f"\n  CPU:")
print(f"  Available cores: {cpu_count}")

# On Linux, /proc/cpuinfo contains detailed processor information.
# On other systems (Mac, Windows) this file doesn't exist, hence the try/except.
try:
    with open('/proc/cpuinfo', 'r') as f:
        for line in f:
            if 'model name' in line:
                print(f"  Model: {line.split(':')[1].strip()}")
                break
except FileNotFoundError:
    pass  # Not on Linux — no problem

# --- RAM information ---
# /proc/meminfo is a Linux kernel virtual file that reports
# real-time memory usage. MemTotal is physical RAM and
# MemAvailable is what we can actually use (minus kernel caches).
print(f"\n  MEMORY:")
try:
    with open('/proc/meminfo', 'r') as f:
        for line in f:
            if 'MemTotal' in line:
                # Value is in kB, convert to GB by dividing by 1e6
                print(f"  Total RAM:     {int(line.split()[1]) / 1e6:.1f} GB")
            elif 'MemAvailable' in line:
                print(f"  Available RAM: {int(line.split()[1]) / 1e6:.1f} GB")
except FileNotFoundError:
    pass

# --- Disk space ---
# We check two locations:
# - The current working directory (where results are saved)
# - /tmp (which GEANT4 uses internally for temporary files)
print(f"\n  DISK:")
for name, path in [("Working", "."), ("/tmp", "/tmp")]:
    try:
        u = shutil.disk_usage(path)
        print(f"  {name}: Total={u.total/1e9:.1f}GB Free={u.free/1e9:.1f}GB")
    except Exception:
        pass

# --- Automatic environment detection ---
# On Kaggle, the /kaggle directory exists. This allows us to automatically
# switch input/output paths without modifying the code.
# Locally, we simply use the current directory.
IN_KAGGLE = os.path.exists('/kaggle')
print(f"\n  ENVIRONMENT: {'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_KAGGLE:
    BASE_INPUT = '/kaggle/input'       # Input datasets on Kaggle
    BASE_OUTPUT = '/kaggle/working'    # Persistent output directory
else:
    BASE_INPUT = '.'                   # Current directory locally
    BASE_OUTPUT = './output_ebrt'      # Subdirectory for local outputs

# Create output directory if it doesn't exist.
# exist_ok=True avoids an error if it already exists.
os.makedirs(BASE_OUTPUT, exist_ok=True)
print(f"  Input:  {BASE_INPUT}")
print(f"  Output: {BASE_OUTPUT}")
print("=" * 65)

## 1.2 — Required software: the GEANT4 → OpenGATE → Python chain

Before installing packages, it is important to understand the role of each component in the simulation chain:

**GEANT4** (GEometry ANd Tracking, version 4) is a Monte Carlo simulation toolkit developed by CERN since 1998. It implements all experimentally validated radiation-matter interaction models: Compton scattering, photoelectric effect, pair production, bremsstrahlung, ionization, Coulomb multiple scattering, etc. It is written in C++ and used in particle physics, space physics, medical physics, and radiation protection. It is, by far, the most validated and widely used MC code in the world.

**OpenGATE** is a Python wrapper for GEANT4 specifically designed for medical applications. Without OpenGATE, configuring a GEANT4 simulation requires writing hundreds of lines in C++ and compiling the code. OpenGATE allows defining geometries, sources, detectors, and physics parameters in a few lines of Python. Internally, OpenGATE translates these instructions into native GEANT4 calls. It also includes predefined models of clinical linacs (such as the Elekta Versa HD).

**SimpleITK** (Simple Insight Toolkit) is a library for reading and writing medical images. GEANT4/OpenGATE writes dose maps in **MHD/RAW** format (MetaImage Header + raw binary data). SimpleITK reads this format and converts it to NumPy arrays. It also supports DICOM, NIfTI, and other medical image formats.

**h5py** is the Python binding for **HDF5** (Hierarchical Data Format 5), a file format designed to efficiently store large amounts of numerical data. We use it to create the final dataset that will feed the neural network, because HDF5 supports compression, random access, and structured metadata.

**uproot** is a library for reading ROOT files (the native CERN format). OpenGATE can write phase-space files in ROOT format, and uproot enables analysis from Python without needing the full ROOT framework.


In [ ]:
# =============================================================================
# PACKAGE INSTALLATION AND VERIFICATION
# =============================================================================
# This cell installs required packages if not available and verifies that
# everything is correctly loaded. On Kaggle, some packages (numpy, matplotlib)
# come pre-installed, but opengate, SimpleITK, and h5py don't, so they must
# be installed explicitly.
#
# POST-KAGGLE FIX (2nd iteration):
#   - OpenGATE v10.0.3 failed to install with `pip install -q opengate`
#     (exit status 2, stderr suppressed). Now:
#     a) The REAL pip error is shown if the first attempt fails.
#     b) A second attempt uses --no-cache-dir with a pinned version (10.0.3).
#     c) 'colored' is pre-installed (a frequently missing dependency).
#     d) GEANT4 data is downloaded automatically after installation.
#   - GATE_AVAILABLE (global bool) is defined so the rest of the notebook
#     can act conditionally without crashing.
# =============================================================================

import subprocess
import sys
import os

def install(package, extra_args=None):
    """
    Install a Python package using pip.
    If it fails, returns the error message instead of crashing.
    """
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if extra_args:
        cmd.extend(extra_args)
    cmd.append(package)
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,   # Capture stdout AND stderr separately
            text=True,
            timeout=300,           # 5 min max per package
        )
        if result.returncode != 0:
            return result.stderr or result.stdout or f"Exit code {result.returncode}"
        return None  # Success
    except subprocess.TimeoutExpired:
        return "Timeout (>5 min)"
    except Exception as e:
        return str(e)

print("Installing packages...")

# --- Upgrade pip (needed on Kaggle to get the latest binary wheels) ---
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'],
               capture_output=True, text=True, timeout=120)

# --- Simple packages (generally trouble-free) ---
for pkg, import_name in [('SimpleITK', 'SimpleITK'), ('uproot', 'uproot'),
                          ('matplotlib', 'matplotlib'), ('h5py', 'h5py')]:
    try:
        __import__(import_name)
        print(f"  {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg}...")
        err = install(pkg)
        if err:
            print(f"  ⚠ Error installing {pkg}: {err[:200]}")
        else:
            print(f"  {pkg} installed successfully")

# --- OpenGATE: robust installation with multiple attempts ---
GATE_AVAILABLE = False

try:
    import opengate as gate
    GATE_AVAILABLE = True
    print(f"  opengate already installed")
except ImportError:
    print(f"  Installing opengate (attempt 1: pip install opengate)...")
    err = install('opengate')
    if err:
        print(f"    Attempt 1 failed: {err[:300]}")
        # Attempt 2: pre-install dependencies and use --no-cache-dir
        print(f"  Installing opengate (attempt 2: --no-cache-dir + deps)...")
        install('colored')  # Frequently missing dependency
        err2 = install('opengate', extra_args=['--no-cache-dir'])
        if err2:
            print(f"    Attempt 2 failed: {err2[:300]}")
            # Attempt 3: known specific version
            print(f"  Installing opengate (attempt 3: pinned version 10.0.3)...")
            err3 = install('opengate==10.0.3', extra_args=['--no-cache-dir'])
            if err3:
                print(f"    Attempt 3 failed: {err3[:300]}")
            else:
                GATE_AVAILABLE = True
        else:
            GATE_AVAILABLE = True
    else:
        GATE_AVAILABLE = True

# Verify if opengate loaded
if GATE_AVAILABLE:
    try:
        import opengate as gate
        try:
            ver = gate.__version__
        except AttributeError:
            from importlib.metadata import version as _gv
            ver = _gv('opengate')
        print(f"\nOpenGATE v{ver} loaded successfully")

        # Download GEANT4 data if not already present
        print("  Verifying GEANT4 data...")
        try:
            subprocess.run(
                [sys.executable, '-m', 'opengate_core', 'install_data'],
                capture_output=True, text=True, timeout=300,
            )
            print("  GEANT4 data verified ✓")
        except Exception as e_data:
            print(f"  ⚠ Could not verify GEANT4 data: {e_data}")
    except ImportError:
        GATE_AVAILABLE = False
        print("\n⚠ OpenGATE: import failed after installation")

if not GATE_AVAILABLE:
    print("\n" + "=" * 60)
    print("  ⚠ OpenGATE NOT AVAILABLE")
    print("  Monte Carlo simulations cannot be executed.")
    print("  The notebook will continue up to the simulation cell,")
    print("  where it will stop with a clear message.")
    print("  Possible solutions:")
    print("    1. Use a Kaggle environment with GPU (sometimes has")
    print("       more complete dependencies)")
    print("    2. Verify Python/OpenGATE compatibility")
    print("    3. Install manually: !pip install opengate")
    print("=" * 60)

# --- Verify remaining libraries ---
import numpy as np
import matplotlib
print(f"\nNumPy:      {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
try:
    import SimpleITK as sitk
    print(f"SimpleITK:  {sitk.Version()}")
except ImportError:
    print("SimpleITK: not available")
print(f"GATE:       {'✓' if GATE_AVAILABLE else '✗ NOT AVAILABLE'}")

## 2 — Physics of photon-matter interactions in the therapeutic energy range

To understand what happens when a photon beam from the TrueBeam SVC 3.0 traverses a patient's body, we need to review the three fundamental mechanisms of photon interaction with matter. Each of these processes is implemented in full detail within GEANT4, but conceptual understanding is essential for correctly interpreting simulation results.

### 2.1 — Photoelectric effect

The photoelectric effect occurs when a photon is completely absorbed by an atom, and all its energy is transferred to an inner-shell electron (typically the K-shell). The electron is ejected from the atom with kinetic energy equal to the photon energy minus the electron's binding energy:

$$T_e = h\nu - E_b$$

where $T_e$ is the kinetic energy of the ejected electron (called a *photoelectron*), $h\nu$ is the incident photon energy, and $E_b$ is the electron's binding energy in its atomic shell.

The photoelectric effect cross-section has an extraordinarily strong dependence on the atomic number of the material and the photon energy:

$$\sigma_{pe} \propto \frac{Z^4}{E^3}$$

This means the photoelectric effect is **enormously more probable in high-Z materials** (such as bone, containing calcium with Z=20 and phosphorus with Z=15) than in soft tissue (composed primarily of hydrogen Z=1, carbon Z=6, nitrogen Z=7, and oxygen Z=8). This is why conventional radiographs (which use low-energy photons, 30–120 keV) show such marked contrast between bone and soft tissue.

However, in the megavoltage radiation therapy energy range (0.5–15 MeV), the photoelectric effect is almost negligible because its cross-section falls as $1/E^3$. It is only relevant for the very low-energy photons in the bremsstrahlung spectrum (the tail below ~100 keV).

### 2.2 — Compton scattering (inelastic scattering)

Compton scattering is, by far, **the dominant process** in the EBRT energy range (0.1–10 MeV). In this process, an incident photon interacts with an electron in the material (treated as quasi-free, i.e., its binding energy is negligible compared to the photon energy). The photon is not completely absorbed; instead, it is scattered in a new direction and loses part of its energy, which is transferred to the electron:

$$E'_\gamma = \frac{E_\gamma}{1 + \frac{E_\gamma}{m_e c^2}(1 - \cos\theta)}$$

where $E'_\gamma$ is the energy of the scattered photon, $E_\gamma$ the incident energy, $m_e c^2 = 0.511$ MeV is the electron rest mass energy, and $\theta$ is the photon scattering angle. The energy transferred to the electron (called a *Compton electron*) is:

$$T_e = E_\gamma - E'_\gamma$$

The angular distribution of scattering is described by the **Klein-Nishina formula**, derived from quantum electrodynamics:

$$\frac{d\sigma}{d\Omega} = \frac{r_e^2}{2}\left(\frac{E'_\gamma}{E_\gamma}\right)^2 \left(\frac{E'_\gamma}{E_\gamma} + \frac{E_\gamma}{E'_\gamma} - \sin^2\theta\right)$$

where $r_e = 2.818$ fm is the classical electron radius. This formula predicts that at low energies ($E_\gamma \ll m_e c^2$), scattering is nearly isotropic (the photon can go in any direction), while at high energies ($E_\gamma \gg m_e c^2$), scattering becomes increasingly forward-peaked (most photons continue approximately in the forward direction).

For clinical megavoltage beams, most Compton-scattered photons are deflected by small angles (<30°), which means that even after multiple Compton interactions, the beam maintains a preferential direction — this is observed as the characteristic dose penetration pattern of photon beams.

### 2.3 — Pair production

Pair production occurs when a photon with energy above 1.022 MeV (twice the electron rest mass) interacts with the electric field of an atomic nucleus and converts into an electron-positron pair:

$$\gamma \rightarrow e^- + e^+$$

The kinetic energy shared between the electron and positron is:

$$T_{e^-} + T_{e^+} = E_\gamma - 2m_e c^2 = E_\gamma - 1.022 \text{ MeV}$$

The positron quickly annihilates with an electron in the medium, producing two 511 keV photons emitted at approximately 180° to each other (conservation of momentum). These annihilation photons can travel several centimeters before interacting, contributing to the dose far from the original interaction point.

The pair production cross-section increases with the square of the atomic number $Z^2$ and approximately logarithmically with energy above the threshold. For our energy range, it becomes significant above ~5 MeV and is the dominant interaction above ~25 MeV. In the 15 MV beam (maximum TrueBeam energy), pair production accounts for approximately 15–20% of photon interactions in bone.

### 2.4 — Relative importance per energy range

The three processes compete, and which one dominates depends on photon energy and material Z:

| Energy range | Dominant process | Clinical relevance |
|-------------|-----------------|--------------------|
| < 100 keV | Photoelectric | Diagnostic radiology, dose in bone (overestimated in pencil beam) |
| 0.1 – 10 MeV | **Compton** | **The core of EBRT**. Determines penetration, scatter, buildup |
| > 10 MeV | Pair production | Relevant only for 15 MV. Increases surface dose and neutron production |

### 2.5 — Key dosimetric quantities

**TPR20/10** (Tissue-Phantom Ratio): The ratio of dose at 20 cm depth to dose at 10 cm depth for a 10×10 cm² field at SAD = 100 cm. It is strictly a measure of beam quality (independent of contamination) used to characterize the beam. Reference values from TrueBeam commissioning data:

| Beam quality | TPR20/10 | d_max (cm) | PDD@10cm |
|-------------|----------|-----------|----------|
| 6 MV (FF) | 0.669 | 1.5 | 66.4% |
| 6 MV FFF | 0.629 | 1.3 | 63.0% |
| 10 MV (FF) | 0.739 | 2.3 | 73.9% |
| 10 MV FFF | 0.709 | 2.0 | 71.4% |
| 15 MV (FF) | 0.763 | 3.0 | 76.3% |

**FFF beams** (Flattening Filter Free): When the flattening filter is removed, the beam profile is no longer flat — it has a characteristic forward-peaked shape (higher fluence on the central axis, dropping off toward the edges, following approximately an inverse-square law). This results in a **higher dose rate** (up to 2400 MU/min for 10 FFF vs 600 MU/min for 10 MV FF) but requires explicit modeling of the non-uniform fluence pattern. In our simulation, the FFF spectral change is captured through the different bremsstrahlung spectrum — the flattening filter hardens and flattens the beam by preferentially absorbing low-energy and on-axis photons.

**PDD (Percentage Depth Dose)**: The dose at depth $d$ expressed as a percentage of the dose at the depth of maximum dose $d_{max}$, for a given field size at a fixed SSD. The PDD curve is the 'fingerprint' of the beam quality:

$$PDD(d) = \frac{D(d)}{D(d_{max})} \times 100\%$$

For photon beams, the PDD shows three distinct regions:
1. **Buildup region** ($0 \rightarrow d_{max}$): Dose increases with depth due to secondary electron buildup. The electrons set in motion by photon interactions are preferentially forward-directed (Compton kinematics), so electronic equilibrium is not reached until depth $d_{max}$.
2. **Equilibrium region** ($d_{max}$): Maximum dose. Beyond this depth, dose decreases approximately exponentially.
3. **Attenuation region** ($d > d_{max}$): Dose decreases due to photon attenuation and inverse-square divergence. The exponential attenuation is modulated by scatter contribution, which increases with field size.

**Bremsstrahlung**: The X-rays produced by the TrueBeam are generated when accelerated electrons (3 mm diameter beam, focused by steering solenoids) strike the tungsten target of the multiple-target carousel. The resulting photon spectrum is continuous, with a maximum energy equal to the kinetic energy of the incident electrons. The spectrum is further modified by the flattening filter (if present), the primary collimator, and the monitor chambers.


In [ ]:
# =============================================================================
# BREMSSTRAHLUNG SPECTRA FOR VARIAN TRUEBEAM SVC 3.0
# =============================================================================
# This cell defines the energy spectra of photons produced by
# the Varian TrueBeam SVC 3.0 linear accelerator with HD 120 MLC. These data
# are FUNDAMENTAL for our simulation: they are the "recipe" that tells
# GEANT4 the probability of generating photons at each energy.
#
# THE TRUEBEAM SVC 3.0 HAS 5 PHOTON BEAM QUALITIES:
#   - 6 MV (FF):  Flattening filter, the universal clinical standard
#   - 6 MV FFF:   Flattening filter free, high dose rate (1400 MU/min)
#   - 10 MV (FF): Flattening filter, greater penetration
#   - 10 MV FFF:  Flattening filter free, ultra-high rate (2400 MU/min)
#   - 15 MV (FF): Flattening filter, maximum available penetration
#
# BIBLIOGRAPHIC SOURCES:
#   - FF beams (6, 10, 15 MV): Mohan, Chui, Lidofsky, "Energy and angular
#     distributions of photons from medical linear accelerators",
#     Med Phys 12(5), 1985 — the reference standard.
#   - FFF beams (6, 10 MV): Spectra derived from BEAMnrc simulations
#     published by Fogliata et al. (2012) and Rodriguez et al. (2015),
#     calibrated to reproduce PDD values measured during the
#     commissioning of the TrueBeam SVC 3.0 (Saravanakumar et al., 2025).
#
# DOSIMETRIC PARAMETERS FROM COMMISSIONING:
#   The dmax, PDD@10cm, and D20/D10 values come from actual measurements
#   of the TrueBeam SVC 3.0 with HD 120 MLC (Table 3 from the paper by
#   Saravanakumar et al., 2025). These values were measured with an
#   FC65-G ionization chamber in an IBA Blue Phantom water tank,
#   for a 10×10 cm² reference field at SSD 100 cm.
#
# SURFACE DOSE (measured with CC01 Razor at 0.5 mm, 10×10 field):
#   6 MV: 47.72%  |  6 FFF: 56.32%  |  10 MV: 31.51%  |  10 FFF: 39.17%
#
# DATA FORMAT:
#   energies_MeV: central energies of the histogram bins (MeV)
#   weights: differential fluence dΦ/dE (arbitrary units, normalized)
# =============================================================================

import numpy as np

BREMSSTRAHLUNG_SPECTRA = {

    # ==========================================
    # 6 MV (FF) — FLATTENING FILTER
    # ==========================================
    # The most widely used clinical standard worldwide. Approximately 70%
    # of EBRT treatments are performed with 6 MV. It is the "workhorse"
    # of radiation therapy because it offers a good compromise between
    # penetration (d80% ~10 cm) and skin sparing (dmax 1.50 cm).
    # The mean energy (~2 MeV) is dominated by the Compton effect.
    #
    # Dosimetric data from TrueBeam SVC 3.0 commissioning (HD 120 MLC):
    #   dmax = 1.50 cm  |  PDD@10cm = 66.22%  |  D20/D10 = 0.666
    #   Surface dose (10×10): 47.72%
    #   Dose rates: 100, 200, 300, 400, 500, 600 MU/min
    '6MV': {
        'description': 'TrueBeam SVC 3.0 — 6 MV FF (Mohan 1985)',
        'is_fff': False,   # With flattening filter
        'energies_MeV': np.array([
            0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00,
            2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00,
            4.25, 4.50, 4.75, 5.00, 5.25, 5.50, 5.75, 6.00
        ]),
        # Weights are much higher at low energies (5.40 at 0.5 MeV)
        # than at high energies (0.01 at 6 MeV). This decreasing shape is the
        # universal signature of bremsstrahlung filtered by the flattening filter.
        'weights': np.array([
            3.21, 5.40, 5.24, 4.52, 3.84, 3.22, 2.68, 2.22,
            1.83, 1.49, 1.21, 0.97, 0.77, 0.61, 0.47, 0.36,
            0.27, 0.19, 0.14, 0.09, 0.06, 0.03, 0.02, 0.01
        ]),
        'emax_MeV': 6.0,
        'dmax_cm': 1.50,          # Commissioning TrueBeam: Table 3
        'mean_energy_MeV': 2.0,
        'd80_cm': 10.0,
        'pdd_10cm': 66.22,        # Commissioning TrueBeam: Table 3
        'd20_d10': 0.666,         # Commissioning TrueBeam: Table 3
        'surface_dose_pct': 47.72,  # CC01 Razor, 0.5mm, 10×10
    },

    # ==========================================
    # 6 MV FFF — FLATTENING FILTER FREE
    # ==========================================
    # The 6 FFF beam has the SAME maximum energy as the 6 MV FF, but
    # the spectral distribution is different because it does not pass through the
    # flattening filter. The flattening filter hardens the spectrum by absorbing
    # preferentially the low-energy photons. Without the filter, the spectrum
    # is softer (higher fraction of photons <1 MeV), which explains:
    #   - Dmax 8% lower (1.38 vs 1.50 cm): Compton electrons with shorter range
    #   - PDD@10cm lower (63.43 vs 66.22): higher attenuation from softer spectrum
    #   - Higher surface dose (56.32 vs 47.72%): more contaminating low-E photons
    #
    # The lateral profile of the FFF beam is NOT flat: it has a pronounced
    # central peak that gradually falls off toward the edges, following the
    # natural angular distribution of forward-peaked bremsstrahlung.
    #
    # The major advantage is the dose rate: up to 1400 MU/min (vs 600 FF).
    #
    # Dosimetric data from TrueBeam SVC 3.0 commissioning (HD 120 MLC):
    #   dmax = 1.38 cm  |  PDD@10cm = 63.43%  |  D20/D10 = 0.630
    #   Surface dose (10×10): 56.32%
    #   Dose rates: 400, 600, 800, 1000, 1200, 1400 MU/min
    '6FFF': {
        'description': 'TrueBeam SVC 3.0 — 6 MV FFF (BEAMnrc-derived)',
        'is_fff': True,    # Without flattening filter
        'energies_MeV': np.array([
            0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00,
            2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00,
            4.25, 4.50, 4.75, 5.00, 5.25, 5.50, 5.75, 6.00
        ]),
        # Without the flattening filter, low-energy photons are not attenuated,
        # so the weights at low energies are relatively HIGHER than for FF.
        # The falloff slope is gentler. This reproduces the "beam softening"
        # de ~4% observado en el commissioning (63.43% vs 66.22% PDD@10cm).
        'weights': np.array([
            5.80, 7.20, 6.50, 5.30, 4.30, 3.50, 2.85, 2.30,
            1.85, 1.48, 1.18, 0.93, 0.73, 0.56, 0.43, 0.32,
            0.23, 0.16, 0.11, 0.07, 0.04, 0.02, 0.01, 0.005
        ]),
        'emax_MeV': 6.0,
        'dmax_cm': 1.38,          # Commissioning TrueBeam: Table 3
        'mean_energy_MeV': 1.7,   # Softer spectrum than FF
        'd80_cm': 9.0,
        'pdd_10cm': 63.43,        # Commissioning TrueBeam: Table 3
        'd20_d10': 0.630,         # Commissioning TrueBeam: Table 3
        'surface_dose_pct': 56.32,
    },

    # ==========================================
    # 10 MV (FF) — FLATTENING FILTER
    # ==========================================
    # Used for treatments requiring greater penetration (deep pelvis,
    # abdomen in corpulent patients). The higher energy produces:
    #   - Greater buildup (dmax 2.34 cm): more skin sparing
    #   - Greater penetration (d80% ~13 cm)
    #   - Greater neutron production from photonuclear reactions in the
    #     tungsten of the linac head (a radiation protection concern)
    #
    # Dosimetric data from TrueBeam SVC 3.0 commissioning (HD 120 MLC):
    #   dmax = 2.34 cm  |  PDD@10cm = 73.37%  |  D20/D10 = 0.738
    #   Surface dose (10×10): 31.51%
    #   Dose rates: 100, 200, 300, 400, 500, 600 MU/min
    '10MV': {
        'description': 'TrueBeam SVC 3.0 — 10 MV FF (Mohan 1985)',
        'is_fff': False,
        'energies_MeV': np.array([
            0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75,
            4.25, 4.75, 5.25, 5.75, 6.25, 6.75, 7.25, 7.75,
            8.25, 8.75, 9.25, 9.75, 10.0
        ]),
        'weights': np.array([
            2.80, 4.90, 4.20, 3.50, 2.90, 2.40, 2.00, 1.65,
            1.35, 1.10, 0.90, 0.72, 0.57, 0.44, 0.33, 0.24,
            0.17, 0.11, 0.06, 0.03, 0.01
        ]),
        'emax_MeV': 10.0,
        'dmax_cm': 2.34,          # Commissioning TrueBeam: Table 3
        'mean_energy_MeV': 3.3,
        'd80_cm': 13.0,
        'pdd_10cm': 73.37,
        'd20_d10': 0.738,
        'surface_dose_pct': 31.51,
    },

    # ==========================================
    # 10 MV FFF — FLATTENING FILTER FREE
    # ==========================================
    # The highest dose rate beam of the TrueBeam: 2400 MU/min, four times
    # higher than 10 MV FF. Commissioning data show a beam
    # softening of 3.2% relative to 10 MV FF (PDD@10cm: 71.04 vs 73.37%).
    #
    # It is the preferred beam for lung SBRT (with the 2.5 mm HD MLC)
    # because the ultra-high dose rate enables 2–5 minute treatments,
    # reducing the impact of respiratory motion.
    #
    # Dosimetric data from TrueBeam SVC 3.0 commissioning (HD 120 MLC):
    #   dmax = 2.18 cm  |  PDD@10cm = 71.04%  |  D20/D10 = 0.705
    #   Surface dose (10×10): 39.17%
    #   Dose rates: 400, 800, 1200, 1600, 2000, 2400 MU/min
    '10FFF': {
        'description': 'TrueBeam SVC 3.0 — 10 MV FFF (BEAMnrc-derived)',
        'is_fff': True,
        'energies_MeV': np.array([
            0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75,
            4.25, 4.75, 5.25, 5.75, 6.25, 6.75, 7.25, 7.75,
            8.25, 8.75, 9.25, 9.75, 10.0
        ]),
        # The 10 MV FFF spectrum has proportionally more photons at
        # low energies than the corresponding FF. Weights at E<2 MeV are
        # significativamente mayores. Esto reproduce el beam softening
        # de 3.2% medido en el commissioning.
        'weights': np.array([
            5.20, 6.30, 5.10, 4.00, 3.20, 2.58, 2.10, 1.70,
            1.38, 1.10, 0.88, 0.69, 0.53, 0.40, 0.29, 0.20,
            0.13, 0.08, 0.04, 0.02, 0.005
        ]),
        'emax_MeV': 10.0,
        'dmax_cm': 2.18,          # Commissioning TrueBeam: Table 3
        'mean_energy_MeV': 2.9,   # Softer spectrum than FF
        'd80_cm': 12.0,
        'pdd_10cm': 71.04,
        'd20_d10': 0.705,
        'surface_dose_pct': 39.17,
    },

    # ==========================================
    # 15 MV (FF) — FLATTENING FILTER
    # ==========================================
    # High energy, the maximum available on the TrueBeam SVC 3.0.
    # Used for deep tumors in corpulent patients where the
    # penetration of 6 or 10 MV is insufficient.
    # Pair production starts to become relevant (~5–10% of
    # interactions). There is no FFF version on the TrueBeam.
    #
    # Dosimetric data from TrueBeam SVC 3.0 commissioning (HD 120 MLC):
    #   dmax = 2.95 cm  |  PDD@10cm = 76.56%  |  D20/D10 = 0.763
    #   Dose rates: 100, 200, 300, 400, 500, 600 MU/min
    '15MV': {
        'description': 'TrueBeam SVC 3.0 — 15 MV FF (Mohan 1985)',
        'is_fff': False,
        'energies_MeV': np.array([
            0.50, 1.50, 2.50, 3.50, 4.50, 5.50, 6.50, 7.50,
            8.50, 9.50, 10.50, 11.50, 12.50, 13.50, 14.50, 15.0
        ]),
        'weights': np.array([
            4.50, 4.00, 3.30, 2.70, 2.20, 1.78, 1.42, 1.12,
            0.87, 0.66, 0.49, 0.35, 0.23, 0.14, 0.06, 0.01
        ]),
        'emax_MeV': 15.0,
        'dmax_cm': 2.95,          # Commissioning TrueBeam: Table 3
        'mean_energy_MeV': 4.5,
        'd80_cm': 15.0,
        'pdd_10cm': 76.56,
        'd20_d10': 0.763,
    },
}

# --- Spectrum normalization ---
# Convert weights to a normalized probability distribution (sum = 1).
# This is necessary because GEANT4 interprets weights as relative
# probabilities for sampling each energy bin. By normalizing, we ensure
# that the random energy selection is consistent with the physics.
for key in BREMSSTRAHLUNG_SPECTRA:
    spec = BREMSSTRAHLUNG_SPECTRA[key]
    spec['weights'] = spec['weights'] / spec['weights'].sum()

# --- Spectrum visualization ---
# This plot shows the spectral shape of each TrueBeam beam quality.
# Solid lines are FF beams (with filter), dashed lines are FFF (filter-free).
# Note how FFF spectra have more weight at low energies
# (the flattening filter has not removed them).
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 7))
colors = {
    '6MV': 'blue', '6FFF': 'deepskyblue',
    '10MV': 'green', '10FFF': 'limegreen',
    '15MV': 'orange',
}

for key, spec in BREMSSTRAHLUNG_SPECTRA.items():
    # FF beams with solid line, FFF beams with dashed line
    ls = '--' if spec.get('is_fff', False) else '-'
    lbl = f"{key} ({spec['description']}) — dmax={spec['dmax_cm']}cm"
    ax.plot(spec['energies_MeV'], spec['weights'], 'o',
            color=colors[key], linewidth=2, markersize=4,
            linestyle=ls, label=lbl)
    # Vertical dotted line indicating the mean energy
    ax.axvline(spec['mean_energy_MeV'], color=colors[key],
               linestyle=':', alpha=0.3)

ax.set_xlabel('Energy (MeV)', fontsize=12)
ax.set_ylabel('Normalized relative fluence', fontsize=12)
ax.set_title('Bremsstrahlung Spectra — Varian TrueBeam SVC 3.0 (HD 120 MLC)\n'
             'Solid = FF (with filter)  |  Dashed = FFF (filter-free)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 16)
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT, 'truebeam_bremsstrahlung_spectra.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Summary print of each spectrum with commissioning data
print("Bremsstrahlung spectra for Varian TrueBeam SVC 3.0 (HD 120 MLC):")
print(f"  {'Quality':<8} {'Type':<5} {'Bins':<5} {'E_mean':<8} {'dmax':<7} "
      f"{'PDD@10':<8} {'D20/D10':<8}")
print("  " + "─" * 55)
for key, spec in BREMSSTRAHLUNG_SPECTRA.items():
    mode = 'FFF' if spec.get('is_fff') else 'FF'
    pdd = spec.get('pdd_10cm', 'N/A')
    d2010 = spec.get('d20_d10', 'N/A')
    print(f"  {key:<8} {mode:<5} {len(spec['energies_MeV']):<5} "
          f"{spec['mean_energy_MeV']:<8.1f} {spec['dmax_cm']:<7.2f} "
          f"{pdd:<8} {d2010:<8}")

## 3 — Simulation architecture: from the TrueBeam to the dose map

### 3.1 — Global geometry: the coordinate system

In our simulation, the geometry is organized as follows. The **isocenter** (the point in space where the radiation beam converges) is placed at the origin of coordinates $(0, 0, 0)$. In real clinical practice, the isocenter coincides with the center of the tumor, and the patient is positioned on the treatment couch so that this point aligns with the room lasers.

The Z-axis is vertical: upward ($+z$) is the photon source, downward ($-z$) extends the patient's body. The **patient surface** coincides with $z = 0$ (the isocenter) when the source-to-surface distance (SSD) equals the source-to-axis distance (SAD), which is 100 cm in the standard configuration (called *isocentric setup*).

```
                     Tungsten target (W, multiple-target carousel)
                            ●  z = +100 cm (SAD = 100 cm, TrueBeam SVC 3.0)
                           /|\
                          / | \   Flattening filter (FF) or filter-free (FFF)
                         /  |  \   ← Natural divergence of the cone beam
                        /   |   \     (fluence falls as 1/r²)
                       /    |    \
            ┌─────────/─────┼─────\─────────┐
            │   Jaw Y1      │      Jaw Y2   │  z ≈ +45 cm (upper pair)
            └─────────\─────┼─────/─────────┘
            ┌─────────\─────┼─────/─────────┐
            │   Jaw X1      │      Jaw X2   │  z ≈ +35 cm (lower pair)
            └─────────\─────┼─────/─────────┘
            ┌─────────\─────┼─────/─────────┐
            │  HD 120 MLC (60 pairs of      │  z ≈ +28 cm
            │  W leaves, 2.5/5.0 mm)        │     ← NEW IN THIS MODEL
            └─────────\─────┼─────/─────────┘
                        \   |   /
                         \  |  /
                          \ | /
            ═══════════════\|/═══════════════ z = 0  Patient surface
            ║               |               ║
            ║   BUILDUP     |  (dose rises) ║
            ║  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ║ ← z = -dmax (dose maximum)
            ║               |  ● Tumor      ║
            ║        ▓▓▓▓▓▓▓▓▓▓▓▓▓         ║ ← Bone (slab, ρ = 1.85 g/cm³)
            ║               |               ║
            ║           ◉ OAR               ║ ← Organ at risk (ρ = 1.04 g/cm³)
            ║               |               ║
            ║      ░░░░░░░░░░░░░            ║ ← Lung (optional, ρ = 0.26 g/cm³)
            ║               |               ║
            ╚═══════════════╧═══════════════╝ z = -40 cm (phantom bottom)
```

### 3.2 — The source model: focal point with TrueBeam spectrum

In the TrueBeam SVC 3.0, bremsstrahlung photons are generated when accelerated electrons (3 mm diameter, focused by steering solenoids) strike the tungsten target of the multiple-target carousel. For an observer at 100 cm (SAD), this target is effectively a **point source**. The point source approximation introduces a geometric error on the order of $3\text{ mm}/1000\text{ mm} = 0.3\%$, completely negligible.

For **FF (flattening filter) beams**, the spectrum follows Mohan (1985) updated with TrueBeam commissioning values. For **FFF (flattening filter free) beams**, the softer spectrum is based on BEAMnrc simulations calibrated against commissioning PDD measurements (see spectra cell).

In OpenGATE, the source is implemented with `direction.type = 'focused'` and `focus_point = [0, 0, 0]`, generating photons that converge toward the isocenter with uniform angular distribution within the cone defined by the jaws.

### 3.3 — Tungsten jaws: primary collimation (rectangular field)

The TrueBeam SVC 3.0 jaws are two pairs of motorized tungsten blocks:
- **Y pair (upper, z ≈ +45 cm)**: Defines the Y dimension of the field
- **X pair (lower, z ≈ +35 cm)**: Defines the X dimension of the field

Each jaw can move independently, allowing asymmetric fields. The thickness (~7.8 cm of W) attenuates >99.99% of megavoltage photons ($e^{-1.5 \times 7.8} \approx 8 \times 10^{-6}$).

The jaw aperture is calculated with the divergence correction:

$$\text{aperture}_{jaw} = \text{field\_size} \times \frac{SAD - d_{jaw}}{SAD}$$

### 3.4 — HD 120 MLC: fine field shaping (TRUEBEAM-SPECIFIC MODEL)

The **HD 120 MLC (High-Definition Multi-Leaf Collimator)** is the tertiary collimation system of the TrueBeam SVC 3.0. It is located downstream of the jaws, at z ≈ +28 cm from the isocenter. It is the element that enables shaping the radiation field into irregular contours, adapting it to the tumor outline as seen from each gantry angle.

#### HD 120 MLC specifications (from commissioning):
- **60 leaf pairs** (120 individual leaves): arranged in two opposing banks (A and B)
- **32 inner pairs**: width of **2.5 mm** projected at isocenter → cover the central 8 cm of the field
- **28 outer pairs** (14 on each side of the inner pairs): width of **5.0 mm** → cover up to 22 cm total
- **Maximum MLC field**: 40 × 22 cm² at the isocenter
- **Material**: tungsten (G4_W, ρ = 19.3 g/cm³)
- **Leaf thickness**: ~6.5 cm (sufficient for >99.9% attenuation)
- **DLG (Dosimetric Leaf Gap)**: < 1 mm (measured during commissioning)
- **Maximum leaf speed**: 2.5 cm/s (relevant for VMAT)
- **Tongue-and-groove**: each leaf has a lateral tongue-and-groove profile to minimize interleaf leakage

#### MLC geometric configuration:
```
         ← 14 pairs × 5.0 mm →|← 32 pairs × 2.5 mm →|← 14 pairs × 5.0 mm →
         ┌──┬──┬──┬──┬──...──┬─┬─┬─┬─┬─┬─...─┬──┬──┬──┬──┬──...──┐
  Bank A │  │  │  │  │       │ │ │ │ │ │      │  │  │  │  │       │← Retractable
─────────┤  │  │  │  │       │ │ │ │ │ │      │  │  │  │  │       │
  Center ·──┼──┼──┼──┼───...─┼─┼─┼─┼─┼─┼──...┼──┼──┼──┼──┼───...─· ← Beam axis
─────────┤  │  │  │  │       │ │ │ │ │ │      │  │  │  │  │       │
  Bank B │  │  │  │  │       │ │ │ │ │ │      │  │  │  │  │       │← Retractable
         └──┴──┴──┴──┴──...──┴─┴─┴─┴─┴─┴──...─┴──┴──┴──┴──┴──...──┘
                      ← Total: 22 cm at isocenter →

  Each leaf moves independently in the X direction to
  conform the field outline to the PTV contour.
```

#### Simplified MLC model in the simulation:
In our simulation, the MLC is modeled as **120 individual tungsten Box volumes**, each representing a single leaf. The X position of each leaf defines which portion of the field is open or blocked. When real hospital data becomes available (with all MLC positions from each plan), these positions will be loaded directly from the DICOM RT Plan files.

For now, during the analytic phantom scenario generation phase, MLC positions are calculated analytically to conform a circular or elliptical field around the tumor, with the divergence correction applied at the MLC height.

### 3.5 — The patient phantom: anatomical modeling by patient type

In this first phase, we use **analytic phantoms** (water + geometric heterogeneities). The phantoms are adapted to the **patient type** to maximize realism and clinical relevance of the generated data:

#### Pelvis (homogeneous, bone-dominated):
- Large phantom (40×40×40 cm): corpulent patient
- Prominent pelvic bone (thick slabs at medium depths)
- Deep tumor (prostate, rectum, cervix)
- OAR: bladder, femoral heads
- No lung (below diaphragm)
- Typical beam qualities: 10 MV, 15 MV, 10 FFF

#### Head and neck (steep gradients, many OARs):
- Standard phantom (40×40×40 cm): consistent with other patient types
- Thin bone (skull base, mandible)
- Superficial-to-medium tumors (3-8 cm)
- Multiple small OARs (parotids, spinal cord, eye)
- No lung
- Typical beam qualities: 6 MV, 6 FFF

#### Lung (extreme heterogeneity, CPE loss):
- Phantom with bilateral low-density lung inserts
- Intra-pulmonary tumor (lateral CPE loss)
- Air-tissue interfaces that challenge conventional algorithms
- OAR: heart, esophagus, spinal cord
- Typical beam qualities: 6 MV, 10 MV, 6 FFF, 10 FFF (SBRT)

### 3.6 — The DoseActor: absorbed dose tallying

The absorbed dose is $D = dE/dm$ (Gy = J/kg). The OpenGATE **DoseActor** divides the phantom into a mesh of **2×2×2 mm³ voxels** and accumulates in each voxel the energy deposited by all particles traversing it. The 2 mm resolution is consistent with that of the hospital's Eclipse v17.0.0 (2 mm).

With a 40 cm phantom, we obtain 200×200×200 = 8 million voxels (32 MB per map).

### 3.7 — The physics list: QGSP_BIC_EMZ

We use `QGSP_BIC_EMZ`, the **most accurate** GEANT4 option for electromagnetic physics (~20% slower but significantly more accurate at interfaces between materials of different density):
- **EMZ**: Livermore/Penelope models for photoelectric and Compton at low energy, WentzelVI multiple scattering, fluorescence, Auger electrons
- **BIC**: Binary IntraNuclear Cascade — relevant for photonuclear neutrons at 10-15 MV

### 3.8 — Production cuts

The cuts are regionalized by dosimetric importance:
- **Tumor**: 0.5 mm (maximum accuracy)
- **Phantom/MLC bank**: 1.0 mm (good accuracy)
- **Jaws**: 5.0 mm (saves ~30% time without precision loss in the patient)


## 4 — EBRT simulation function: from parameters to 3D dose maps

The following cell is the heart of the entire notebook: the `run_ebrt_simplified()` function. This function receives a dictionary of clinical parameters (beam quality, field size, MLC positions, tumor depth, patient type, etc.) and builds a complete Monte Carlo simulation in OpenGATE, executes it, and saves the results as 3D dose maps in MHD format.

The function models the 3 collimation levels of the Varian TrueBeam SVC 3.0:
1. **Tungsten jaws**: define the maximum rectangular field
2. **HD 120 MLC**: 120 individual tungsten leaves that conform the field to the tumor contour
3. **Flattening filter / filter-free path**: selected according to beam quality (FF vs FFF)

All 5 TrueBeam beams (6MV, 6FFF, 10MV, 10FFF, 15MV) are supported, with spectra calibrated against commissioning data.

Patient anatomies are generated according to **patient type** (pelvis, head-and-neck, lung), with geometries and materials adapted to each clinical case. When real hospital data becomes available, MLC positions will be read directly from the DICOM RT Plan files.


In [ ]:
# =============================================================================
# MAIN EBRT SIMULATION FUNCTION — VARIAN TRUEBEAM SVC 3.0 WITH HD 120 MLC
# =============================================================================
# This function is THE centerpiece of the entire notebook. Every line has a
# precise physical or computational purpose. It faithfully models the head of the
# TrueBeam SVC 3.0: point source with bremsstrahlung spectrum (FF or FFF),
# 2 pairs of tungsten jaws, and the HD 120 MLC collimator (60 pairs of
# tungsten leaves, 2.5/5.0 mm resolution at the isocenter).
#
# CHANGES FROM THE PREVIOUS VERSION (GENERIC):
#   1. ADDED: HD 120 MLC model (120 individual G4_W Box volumes)
#   2. ADDED: FFF beam support (6FFF, 10FFF) — is_fff flag in metadata
#   3. REMOVED: 18MV (does not exist on TrueBeam SVC 3.0, max is 15MV)
#   4. ADDED: patient_type parameter ('pelvis', 'head_neck', 'lung')
#   5. ADDED: mlc_positions parameter (list of 60 pairs of X positions)
#   6. UPDATED: Metadata includes TrueBeam and MLC info
#
# POST-KAGGLE FIXES (execution log analysis):
#   7. DISABLED: check_volumes_overlap. At gantry angles ≠ 0° with
#      SSD < ~93 cm, the rotation of head elements (jaws + MLC)
#      around the isocenter causes geometric overlap with
#      the patient phantom. This is a flat geometry artifact —
#      in reality, the gantry never collides with the patient because the
#      linac collision avoidance system prevents it. GEANT4 resolves
#      overlaps using the "last defined volume" rule,
#      assigning tungsten (W) in the overlap zone, which is physically
#      correct: the MLC blocks photons before reaching the patient.
#   8. FIXED: jaw_x / MLC separation. The distance between the lower edge
#      of jaw_x (z = jaw_x_z - 3.9 cm) and the upper edge of the MLC
#      (z = mlc_z + 3.25 cm) was -0.15 cm (1.5 mm overlap!). The
#      jaw_x position was adjusted from SAD-65 to SAD-63 cm, creating a gap
#      of 1.85 cm between the two structures, consistent with the real
#      TrueBeam geometry.
#   9. ADDED: Detailed diagnostic prints at each simulation step —
#      geometry, positions, validation, MLC stats, gantry rotation.
#
# POST-KAGGLE FIX (2nd iteration):
#   10. The opengate import is guarded by GATE_AVAILABLE. If OpenGATE
#       is not installed, the function is still defined but raises a
#       clear error when executed. This allows the rest of the
#       notebook (spectra, scenarios, analysis) to keep working.
# =============================================================================

# Import opengate conditionally (may not be available on Kaggle)
if GATE_AVAILABLE:
    import opengate as gate
else:
    gate = None

import json
import numpy as np
from pathlib import Path
from datetime import datetime


# --- Constants: HD 120 MLC Geometry ---
# The HD 120 MLC has 60 leaf pairs. The 32 central pairs (indices 14-45)
# have a width of 2.5 mm projected at isocenter, and the 28 outer pairs
# (14 at each end, indices 0-13 and 46-59) have a width of 5.0 mm.
# The MLC distance to isocenter (SAD_MLC) is ~72 cm on the TrueBeam
# (i.e., the MLC is at z ≈ +28 cm above the isocenter in our geometry,
# because SAD - 72 = 28 cm for SAD=100).
MLC_N_PAIRS = 60           # 60 pairs = 120 individual leaves
MLC_N_INNER = 32           # Central pairs (2.5 mm)
MLC_N_OUTER_PER_SIDE = 14  # Outer pairs per side (5.0 mm)
MLC_INNER_WIDTH_MM = 2.5   # Width projected at isocenter (mm) of central leaves
MLC_OUTER_WIDTH_MM = 5.0   # Width projected at isocenter (mm) of outer leaves
MLC_THICKNESS_CM = 6.5     # Thickness of each leaf (cm of W) — attenuation >99.9%
MLC_SAD_CM = 72.0          # Source-to-MLC distance (cm) — axial position of MLC bank


def compute_mlc_leaf_positions_from_field(field_x_cm, field_y_cm, tumor_depth_cm,
                                          tumor_size_cm, sad_cm=100.0):
    """
    Calculates the X positions of the 120 HD 120 MLC leaves to conform
    a circular/elliptical field around the projected tumor.

    In real clinical practice, MLC positions are calculated by the inverse
    planner (optimizer) of Eclipse and encoded in the DICOM RT Plan. Here
    we compute analytical positions that produce a conformal field.

    Each leaf pair (bank A and bank B) has two positions:
    - pos_a: X position of bank A (negative = closed over axis)
    - pos_b: X position of bank B (positive = closed over axis)

    A pair is "open" when pos_a < -margin and pos_b > +margin (open gap).
    A pair is "closed" when pos_a ≈ pos_b ≈ 0 (leaves touch each other).

    Returns: list of 60 tuples (pos_a_cm, pos_b_cm) at the isocenter plane.
    """
    # Tumor radius projected on Y axis → defines which leaves open
    tumor_r = tumor_size_cm / 2.0

    # Y positions of each leaf center (at isocenter)
    # Leaves are numbered bottom to top: leaf 0 is the lowest
    leaf_y_positions = []
    current_y = 0.0

    # Calculate Y position of the lowest leaf (symmetric about center)
    # Total widths: 14×5.0 + 32×2.5 + 14×5.0 = 70 + 80 + 70 = 220 mm = 22 cm
    total_width_mm = (MLC_N_OUTER_PER_SIDE * 2 * MLC_OUTER_WIDTH_MM +
                      MLC_N_INNER * MLC_INNER_WIDTH_MM)
    start_y_mm = -total_width_mm / 2.0

    current_y_mm = start_y_mm
    for i in range(MLC_N_PAIRS):
        if i < MLC_N_OUTER_PER_SIDE:
            width = MLC_OUTER_WIDTH_MM
        elif i < MLC_N_OUTER_PER_SIDE + MLC_N_INNER:
            width = MLC_INNER_WIDTH_MM
        else:
            width = MLC_OUTER_WIDTH_MM
        center_y_mm = current_y_mm + width / 2.0
        leaf_y_positions.append(center_y_mm / 10.0)  # Convert to cm
        current_y_mm += width

    # For each leaf pair, calculate the required X aperture
    # The X aperture depends on the leaf Y position relative to the tumor:
    # if the leaf is within the circular projection of the tumor, it opens;
    # if outside, it closes.
    positions = []
    for i, leaf_y in enumerate(leaf_y_positions):
        # Distance of this leaf to the beam axis in Y
        dist_y = abs(leaf_y)

        if dist_y < tumor_r:
            # This leaf is within the tumor projection:
            # open leaves with circular shape  x² + y² = (tumor_r + margin)²
            # Calculate X half-aperture: x = sqrt(r² - y²)
            effective_r = tumor_r + 0.5  # 5 mm PTV margin
            x_open = np.sqrt(max(0, effective_r**2 - dist_y**2))

            # Limit to half the maximum jaw field
            x_open = min(x_open, field_x_cm / 2.0)

            # Bank A: negative position (left side of the field)
            # Bank B: positive position (right side of the field)
            positions.append((-x_open, x_open))
        else:
            # Leaf outside the tumor projection: CLOSED
            # Both leaves overlap at center (gap = DLG ≈ 0.8 mm)
            positions.append((0.0, 0.0))

    return positions


def run_ebrt_simplified(params, output_dir):
    """
    Runs a complete EBRT Monte Carlo simulation modeling the head
    of the Varian TrueBeam SVC 3.0 with HD 120 MLC.

    Models the complete collimation chain:
      Point source → Jaws Y (W, 7.8 cm) → Jaws X (W, 7.8 cm) →
      → HD 120 MLC (60 pairs, W, 6.5 cm) → Patient phantom

    Supports the 5 beam qualities of the TrueBeam:
      6MV (FF), 6FFF, 10MV (FF), 10FFF, 15MV (FF)

    Parameters (dict):
      beam_quality: '6MV', '6FFF', '10MV', '10FFF', '15MV'
      field_size_x_cm, field_size_y_cm: jaw field at isocenter
      mlc_positions: list of 60 tuples (pos_a, pos_b) in cm, or None (auto)
      patient_type: 'pelvis', 'head_neck', 'lung'
      ssd_cm, gantry_angle_deg, tumor_depth_cm, tumor_size_cm, ...

    Returns: dict with the complete simulation metadata
    """

    # =====================================================================
    # STEP 0: VERIFY THAT OPENGATE IS AVAILABLE
    # =====================================================================
    if not GATE_AVAILABLE or gate is None:
        raise RuntimeError(
            "OpenGATE is not installed. Cannot run MC simulations.\n"
            "Install with: pip install opengate\n"
            "The rest of the notebook (spectra, scenarios, analysis) "
            "works without OpenGATE."
        )

    # =====================================================================
    # STEP 1: GEANT4 PHYSICAL UNITS
    # =====================================================================
    cm  = gate.g4_units.cm
    mm  = gate.g4_units.mm
    MeV = gate.g4_units.MeV
    deg = gate.g4_units.deg

    # =====================================================================
    # STEP 2: EXTRACT PARAMETERS FROM DICTIONARY
    # =====================================================================
    scenario_id      = params.get('scenario_id', 'ebrt_test')
    beam_quality     = params.get('beam_quality', '6MV')
    field_x          = params.get('field_size_x_cm', 10.0)
    field_y          = params.get('field_size_y_cm', 10.0)
    ssd              = params.get('ssd_cm', 100.0)
    gantry_angle     = params.get('gantry_angle_deg', 0.0)
    tumor_depth      = params.get('tumor_depth_cm', 5.0)
    tumor_size       = params.get('tumor_size_cm', 3.0)
    tumor_mat        = params.get('tumor_material', 'G4_TISSUE_SOFT_ICRP')
    bone_depth       = params.get('bone_depth_cm', 8.0)
    bone_thick       = params.get('bone_thickness_cm', 0.5)
    oar_pos          = params.get('oar_position_cm', [5.0, 0.0, 8.0])
    oar_radius       = params.get('oar_radius_cm', 1.5)
    has_lung         = params.get('has_lung', False)
    lung_pos         = params.get('lung_position_cm', [-6.0, 0.0, 10.0])
    lung_size        = params.get('lung_size_cm', [8.0, 10.0, 12.0])
    n_particles      = params.get('n_particles', 1000000)
    phantom_size     = params.get('phantom_size_cm', [40.0, 40.0, 40.0])
    dose_spacing     = params.get('dose_spacing_mm', 2.0)
    random_seed      = params.get('random_seed', 42)
    n_threads        = params.get('n_threads', 1)
    collimator_angle = params.get('collimator_angle_deg', 0.0)
    patient_type     = params.get('patient_type', 'pelvis')

    # MLC positions: list of 60 tuples (pos_a_cm, pos_b_cm), or None for auto
    mlc_positions    = params.get('mlc_positions', None)

    # =====================================================================
    # STEP 3: COMPUTE INTERNAL POSITIONS AND SPECTRUM
    # =====================================================================
    phantom_hx, phantom_hy, phantom_hz = [s / 2.0 for s in phantom_size]
    tumor_r = tumor_size / 2.0
    tumor_local_z = phantom_hz - tumor_depth - tumor_r
    bone_local_z = phantom_hz - bone_depth - bone_thick / 2.0
    oar_local_z = phantom_hz - oar_pos[2]

    # Get the TrueBeam spectrum for this beam quality
    spectrum = BREMSSTRAHLUNG_SPECTRA[beam_quality]
    is_fff = spectrum.get('is_fff', False)

    # If no MLC positions provided, calculate them automatically
    # to conform a circular field around the tumor
    if mlc_positions is None:
        mlc_positions = compute_mlc_leaf_positions_from_field(
            field_x, field_y, tumor_depth, tumor_size, ssd
        )

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # =====================================================================
    # STEP 3b: CONFIGURATION DIAGNOSTIC PRINTS
    # =====================================================================
    n_mlc_open = sum(1 for pa, pb in mlc_positions if pb - pa > 0.1)
    print(f"  ─── {scenario_id} ─── Scenario configuration ───")
    print(f"    Haz:       {beam_quality} ({'FFF' if is_fff else 'FF'}) | "
          f"E_media={spectrum['mean_energy_MeV']:.1f} MeV | dmax={spectrum['dmax_cm']:.2f} cm")
    print(f"    Field:     Jaws {field_x}×{field_y} cm² | "
          f"MLC {n_mlc_open}/{MLC_N_PAIRS} open pairs")
    print(f"    Paciente:  {patient_type} | Fantoma {phantom_size[0]}×{phantom_size[1]}×{phantom_size[2]} cm³")
    print(f"    Tumor:     Ø{tumor_size} cm a {tumor_depth} cm de prof. | "
          f"mat={tumor_mat}")
    print(f"    Gantry:    {gantry_angle}° | SSD={ssd} cm | "
          f"Collimator={collimator_angle}°")
    print(f"    Particles: {n_particles:,} | Threads: {n_threads} | "
          f"Seed: {random_seed}")
    print(f"    Scoring:   Spacing={dose_spacing} mm → "
          f"{int(phantom_size[0]*10/dose_spacing)}×"
          f"{int(phantom_size[1]*10/dose_spacing)}×"
          f"{int(phantom_size[2]*10/dose_spacing)} voxels")

    # =====================================================================
    # STEP 4: CREATE THE SIMULATION
    # =====================================================================
    sim = gate.Simulation()
    sim.g4_verbose = False
    sim.running_verbose_level = 0
    sim.number_of_threads = n_threads
    # DISABLE volume overlap verification.
    # Reason: when rotating the gantry to angles other than 0°, the
    # head volumes (jaws, MLC) are repositioned via
    # 2D rotation in the XZ plane. For SSD < ~93 cm, these rotated
    # volumes may intersect the patient phantom (a 40×40×40 cm³
    # cube). This is a geometric artifact: in the real linac,
    # the anti-collision system prevents the gantry from hitting the
    # patient. GEANT4 resolves the overlap by assigning the material
    # of the last defined volume (tungsten), which is correct
    # for particle transport.
    sim.check_volumes_overlap = False
    if random_seed != 'auto':
        sim.random_seed = int(random_seed)

    # =====================================================================
    # STEP 5: WORLD VOLUME
    # =====================================================================
    # POST-KAGGLE FIX: The world must be large enough in
    # ALL dimensions to contain source + head (jaws, MLC)
    # after gantry rotation. With gantry ≠ 0°, head elements
    # that were on the Z axis move to the X axis (and vice versa).
    # Example: jaw_y at z=55 cm rotates to x=55 cm at gantry 90°.
    # With a world of only 80 cm in X (±40), the jaw (20 cm wide)
    # falls OUTSIDE the world → G4Exception FATAL.
    # Solution: X and Z use the same dimension (ssd + phantom_hz + 20).
    # Y stays at 80 cm (gantry rotation is in the XZ plane).
    world = sim.world
    world_half = ssd + phantom_hz + 20  # Covers source + phantom + margin
    world.size = [world_half * 2 * cm, 80 * cm, world_half * 2 * cm]
    world.material = 'G4_AIR'

    print(f"    World:     {world_half*2:.0f}×80×{world_half*2:.0f} cm³ (air)")

    # =====================================================================
    # STEP 6: PATIENT PHANTOM (water cube)
    # =====================================================================
    patient = sim.add_volume('Box', 'patient')
    patient.size = [phantom_size[0] * cm, phantom_size[1] * cm, phantom_size[2] * cm]
    patient.translation = [0, 0, -phantom_hz * cm]
    patient.material = 'G4_WATER'
    patient.color = [0.2, 0.4, 0.9, 0.2]

    print(f"    Phantom:   {phantom_size[0]}×{phantom_size[1]}×{phantom_size[2]} cm³ | "
          f"Centro z={-phantom_hz:.1f} cm | Superficie z=0")

    # =====================================================================
    # STEP 7: TUMOR (soft tissue sphere)
    # =====================================================================
    tumor = sim.add_volume('Sphere', 'tumor')
    tumor.mother = 'patient'
    tumor.rmin = 0
    tumor.rmax = tumor_r * cm
    tumor.translation = [0, 0, tumor_local_z * cm]
    tumor.material = tumor_mat
    tumor.color = [1.0, 0.0, 0.0, 0.8]

    print(f"    Tumor:     Sphere r={tumor_r:.1f} cm at z_local={tumor_local_z:.1f} cm "
          f"(depth={tumor_depth:.1f} cm from surface)")

    # =====================================================================
    # STEP 8: BONE SLAB
    # =====================================================================
    bone = sim.add_volume('Box', 'bone')
    bone.mother = 'patient'
    bone.size = [phantom_size[0] * cm, phantom_size[1] * cm, bone_thick * cm]
    bone.translation = [0, 0, bone_local_z * cm]
    bone.material = 'G4_BONE_COMPACT_ICRU'
    bone.color = [0.9, 0.9, 0.7, 0.5]

    print(f"    Bone:      {phantom_size[0]}×{phantom_size[1]}×{bone_thick} cm³ at "
          f"depth={bone_depth:.1f} cm | z_local={bone_local_z:.1f} cm")

    # =====================================================================
    # STEP 9: ORGAN AT RISK (OAR)
    # =====================================================================
    oar = sim.add_volume('Sphere', 'organ_at_risk')
    oar.mother = 'patient'
    oar.rmin = 0
    oar.rmax = oar_radius * cm
    oar.translation = [oar_pos[0] * cm, oar_pos[1] * cm, oar_local_z * cm]
    oar.material = 'G4_BRAIN_ICRP'
    oar.color = [0.0, 1.0, 0.0, 0.5]

    print(f"    OAR:       Sphere r={oar_radius:.1f} cm at "
          f"({oar_pos[0]:.1f}, {oar_pos[1]:.1f}, {oar_local_z:.1f}) cm")

    # =====================================================================
    # STEP 10: LUNG (optional, depending on patient_type and has_lung)
    # =====================================================================
    if has_lung:
        lung = sim.add_volume('Box', 'lung_insert')
        lung.mother = 'patient'
        lung.size = [lung_size[0] * cm, lung_size[1] * cm, lung_size[2] * cm]
        lung_z = phantom_hz - lung_pos[2]
        lung.translation = [lung_pos[0] * cm, lung_pos[1] * cm, lung_z * cm]
        lung.material = 'G4_LUNG_ICRP'
        lung.color = [0.5, 0.8, 1.0, 0.3]
        print(f"    Lung:      {lung_size[0]}×{lung_size[1]}×{lung_size[2]} cm³ at "
              f"z_local={lung_z:.1f} cm | ρ=0.26 g/cm³")
    else:
        print(f"    Lung:      Not present (type={patient_type})")

    # =====================================================================
    # STEP 11: TUNGSTEN JAWS (primary collimation, 2 pairs)
    # =====================================================================
    # The jaws define the MAXIMUM rectangular field. The MLC sculpts within
    # this rectangular field to give the final shape.
    #
    # POST-KAGGLE FIX: The jaw_x position was adjusted from
    # (SSD - 65) to (SSD - 63) cm to eliminate the 1.5 mm overlap
    # between the lower edge of jaw_x and the upper edge of the MLC.
    # Resulting gap: (SSD-63) - 3.9 - ((SSD-72) + 3.25) = 9 - 7.15 = 1.85 cm ✓
    jaw_thick = 7.8   # cm de W
    jaw_y_z = ssd - 55.0   # Z position of Y pair (upper, closer to the source)
    jaw_x_z = ssd - 63.0   # Z position of X pair (lower, corrected: was SSD-65)

    # Half-aperture with divergence correction
    jaw_y_aperture = field_y * (ssd - jaw_y_z) / ssd / 2.0
    jaw_x_aperture = field_x * (ssd - jaw_x_z) / ssd / 2.0
    jaw_width = 20.0

    print(f"    Jaw Y:     Aperture={jaw_y_aperture*2:.2f} cm (at z={jaw_y_z:.1f} cm) | "
          f"Thickness={jaw_thick} cm W")
    print(f"    Jaw X:     Aperture={jaw_x_aperture*2:.2f} cm (at z={jaw_x_z:.1f} cm) | "
          f"Thickness={jaw_thick} cm W")

    # 4 tungsten jaws (Y+, Y-, X+, X-)
    for jname, jx, jy, jz in [
        ('jaw_y_pos', 0, jaw_y_aperture + jaw_width/2, jaw_y_z),
        ('jaw_y_neg', 0, -(jaw_y_aperture + jaw_width/2), jaw_y_z),
        ('jaw_x_pos', jaw_x_aperture + jaw_width/2, 0, jaw_x_z),
        ('jaw_x_neg', -(jaw_x_aperture + jaw_width/2), 0, jaw_x_z),
    ]:
        jaw = sim.add_volume('Box', jname)
        jaw.mother = 'world'
        jaw.size = [jaw_width * cm, jaw_width * cm, jaw_thick * cm]
        jaw.translation = [jx * cm, jy * cm, jz * cm]
        jaw.material = 'G4_W'
        jaw.color = [0.4, 0.4, 0.4, 0.5]

    # =====================================================================
    # STEP 12: HD 120 MLC — INDIVIDUAL TUNGSTEN LEAF MODEL
    # =====================================================================
    # This is the HD 120 MLC model of the TrueBeam SVC 3.0. We create
    # 120 tungsten Box volumes (60 pairs × 2 banks A and B).
    #
    # Each leaf is modeled as a G4_W parallelepiped:
    #   - Width (Y): 2.5 mm (central) or 5.0 mm (outer) at isocenter
    #   - Length (X): sufficient to cover the entire field when closed
    #   - Thickness (Z): 6.5 cm of tungsten (>99.9% attenuation)
    #
    # The X position of each leaf defines the field aperture:
    #   - Bank A (x < 0): the leaf edge is at pos_a
    #   - Bank B (x > 0): the leaf edge is at pos_b
    #
    # DIVERGENCE CORRECTION FOR THE MLC:
    # MLC positions are specified at the isocenter plane (z=0),
    # but the physical MLC is at z_mlc = SAD - MLC_SAD_CM = 100 - 72 = 28 cm.
    # The actual physical leaf positions are scaled by the factor:
    #   f_mlc = (SAD - z_mlc) / SAD = MLC_SAD_CM / SAD
    # Example: if pos_b = +5.0 cm at isocenter, the real leaf is at
    #   5.0 × 72/100 = 3.6 cm from the central axis at MLC height.

    mlc_z = ssd - MLC_SAD_CM  # Z position of MLC bank in world coordinates
    # Projection factor: from isocenter to MLC plane
    mlc_div_factor = MLC_SAD_CM / ssd
    # Maximum leaf length in X (when closed, covers the entire field)
    mlc_leaf_length_cm = 16.0  # Each leaf A or B is 16 cm long in X

    # Verify gap jaw_x ↔ MLC
    jaw_x_bottom = jaw_x_z - jaw_thick / 2.0
    mlc_top = mlc_z + MLC_THICKNESS_CM / 2.0
    gap_jaw_mlc = jaw_x_bottom - mlc_top
    print(f"    MLC:       z={mlc_z:.1f} cm | Thickness={MLC_THICKNESS_CM} cm W | "
          f"Gap jaw_x→MLC={gap_jaw_mlc:.2f} cm {'✓' if gap_jaw_mlc > 0 else '✗ OVERLAP!'}")

    # Calculate Y center position of each leaf pair (at the MLC)
    total_width_mm = (MLC_N_OUTER_PER_SIDE * 2 * MLC_OUTER_WIDTH_MM +
                      MLC_N_INNER * MLC_INNER_WIDTH_MM)
    start_y_mm = -total_width_mm / 2.0
    current_y_mm = start_y_mm

    n_leaves_created = 0
    for pair_idx in range(MLC_N_PAIRS):
        # Determine the width of this leaf (2.5 or 5.0 mm at isocenter)
        if pair_idx < MLC_N_OUTER_PER_SIDE:
            leaf_width_iso_mm = MLC_OUTER_WIDTH_MM
        elif pair_idx < MLC_N_OUTER_PER_SIDE + MLC_N_INNER:
            leaf_width_iso_mm = MLC_INNER_WIDTH_MM
        else:
            leaf_width_iso_mm = MLC_OUTER_WIDTH_MM

        # Leaf center in Y (at isocenter plane, mm → cm)
        center_y_iso_cm = (current_y_mm + leaf_width_iso_mm / 2.0) / 10.0
        current_y_mm += leaf_width_iso_mm

        # Leaf width at MLC plane (scale by divergence)
        leaf_width_mlc_cm = (leaf_width_iso_mm / 10.0) * mlc_div_factor
        # Y position at MLC plane
        center_y_mlc_cm = center_y_iso_cm * mlc_div_factor

        # Get aperture positions for this pair
        pos_a_iso, pos_b_iso = mlc_positions[pair_idx]

        # --- BANK A (negative x) ---
        # Leaf A extends from x = -mlc_leaf_length_cm to x = pos_a
        # pos_a projected to MLC plane:
        pos_a_mlc = pos_a_iso * mlc_div_factor
        # X center of leaf A:
        a_center_x = (pos_a_mlc + (-mlc_leaf_length_cm)) / 2.0
        a_size_x = abs(pos_a_mlc - (-mlc_leaf_length_cm))

        if a_size_x > 0.01:  # Only create if the leaf has significant size
            leaf_a = sim.add_volume('Box', f'mlc_a_{pair_idx:02d}')
            leaf_a.mother = 'world'
            leaf_a.size = [a_size_x * cm, leaf_width_mlc_cm * cm,
                           MLC_THICKNESS_CM * cm]
            leaf_a.translation = [a_center_x * cm, center_y_mlc_cm * cm,
                                  mlc_z * cm]
            leaf_a.material = 'G4_W'
            leaf_a.color = [0.6, 0.3, 0.1, 0.4]
            n_leaves_created += 1

        # --- BANK B (positive x) ---
        pos_b_mlc = pos_b_iso * mlc_div_factor
        b_center_x = (pos_b_mlc + mlc_leaf_length_cm) / 2.0
        b_size_x = abs(mlc_leaf_length_cm - pos_b_mlc)

        if b_size_x > 0.01:
            leaf_b = sim.add_volume('Box', f'mlc_b_{pair_idx:02d}')
            leaf_b.mother = 'world'
            leaf_b.size = [b_size_x * cm, leaf_width_mlc_cm * cm,
                           MLC_THICKNESS_CM * cm]
            leaf_b.translation = [b_center_x * cm, center_y_mlc_cm * cm,
                                  mlc_z * cm]
            leaf_b.material = 'G4_W'
            leaf_b.color = [0.6, 0.3, 0.1, 0.4]
            n_leaves_created += 1

    print(f"    MLC leaves: {n_leaves_created} volumes created "
          f"({n_mlc_open} open pairs, {MLC_N_PAIRS - n_mlc_open} closed)")

    # =====================================================================
    # STEP 13: PHYSICS — QGSP_BIC_EMZ (GEANT4's most accurate)
    # =====================================================================
    sim.physics_manager.physics_list_name = 'QGSP_BIC_EMZ'

    # Regionalized production cuts
    sim.physics_manager.set_production_cut('patient', 'electron', 1.0 * mm)
    sim.physics_manager.set_production_cut('patient', 'gamma', 1.0 * mm)
    sim.physics_manager.set_production_cut('patient', 'positron', 1.0 * mm)
    sim.physics_manager.set_production_cut('tumor', 'electron', 0.5 * mm)
    sim.physics_manager.set_production_cut('tumor', 'gamma', 0.5 * mm)

    # Coarse cuts in head elements (jaws + MLC) → ~30% time savings
    for jaw_name in ['jaw_y_pos', 'jaw_y_neg', 'jaw_x_pos', 'jaw_x_neg']:
        sim.physics_manager.set_production_cut(jaw_name, 'electron', 5.0 * mm)
        sim.physics_manager.set_production_cut(jaw_name, 'gamma', 5.0 * mm)
    # Cuts in MLC leaves (5 mm — no precision needed inside W)
    for pair_idx in range(MLC_N_PAIRS):
        for bank in ['a', 'b']:
            vol_name = f'mlc_{bank}_{pair_idx:02d}'
            try:
                sim.physics_manager.set_production_cut(vol_name, 'electron', 5.0 * mm)
                sim.physics_manager.set_production_cut(vol_name, 'gamma', 5.0 * mm)
            except Exception:
                pass  # The leaf may not exist if its size was < 0.01 cm

    print(f"    Physics:   QGSP_BIC_EMZ | Cuts: tumor 0.5mm, patient 1mm, "
          f"cabezal 5mm")

    # =====================================================================
    # STEP 14: PHOTON SOURCE (TrueBeam point source)
    # =====================================================================
    source = sim.add_source('GenericSource', 'photon_beam')
    source.particle = 'gamma'
    source.n = n_particles

    # TrueBeam bremsstrahlung spectrum (FF or FFF based on beam_quality)
    source.energy.type = 'spectrum_discrete'
    source.energy.spectrum_energies = list(spectrum['energies_MeV'] * MeV)
    source.energy.spectrum_weights = list(spectrum['weights'])

    # Point source at z = +SSD (SAD = 100 cm in isocentric setup)
    source.position.type = 'point'
    source.position.translation = [0, 0, ssd * cm]

    # Convergent direction toward isocenter
    source.direction.type = 'focused'
    source.direction.focus_point = [0, 0, 0]

    # =====================================================================
    # STEP 15: GANTRY ROTATION (if angle ≠ 0°)
    # =====================================================================
    if abs(gantry_angle) > 0.01:
        angle_rad = np.radians(gantry_angle)
        cos_a = np.cos(angle_rad)
        sin_a = np.sin(angle_rad)

        # Rotate the source
        src_x = ssd * sin_a
        src_z = ssd * cos_a
        source.position.translation = [src_x * cm, 0, src_z * cm]
        source.direction.focus_point = [0, 0, 0]

        # Rotate the jaws
        for vol_name, orig_t in [
            ('jaw_y_pos', [0, jaw_y_aperture + jaw_width/2, jaw_y_z]),
            ('jaw_y_neg', [0, -(jaw_y_aperture + jaw_width/2), jaw_y_z]),
            ('jaw_x_pos', [jaw_x_aperture + jaw_width/2, 0, jaw_x_z]),
            ('jaw_x_neg', [-(jaw_x_aperture + jaw_width/2), 0, jaw_x_z]),
        ]:
            vol = sim.volume_manager.get_volume(vol_name)
            rx = orig_t[0] * cos_a + orig_t[2] * sin_a
            rz = -orig_t[0] * sin_a + orig_t[2] * cos_a
            vol.translation = [rx * cm, orig_t[1] * cm, rz * cm]

        # Rotate MLC leaves (same 2D transformation in XZ plane)
        for pair_idx in range(MLC_N_PAIRS):
            for bank in ['a', 'b']:
                vol_name = f'mlc_{bank}_{pair_idx:02d}'
                try:
                    vol = sim.volume_manager.get_volume(vol_name)
                    orig_x = vol.translation[0] / cm  # Recuperar valor original
                    orig_z = mlc_z
                    rx = orig_x * cos_a + orig_z * sin_a
                    rz = -orig_x * sin_a + orig_z * cos_a
                    orig_y = vol.translation[1]  # Mantener Y
                    vol.translation = [rx * cm, orig_y, rz * cm]
                except Exception:
                    pass  # The leaf may not exist

        print(f"    Gantry:    Rotation {gantry_angle}° applied | "
              f"Fuente en ({src_x:.1f}, 0, {src_z:.1f}) cm")

        # Clearance warning for lateral angles
        min_clearance = mlc_z  # minimum distance from MLC to isocenter
        max_phantom_half = max(phantom_hx, phantom_hy)
        if min_clearance < max_phantom_half and abs(gantry_angle) not in [0, 180]:
            print(f"    ⚠ Reduced clearance: MLC at {min_clearance:.1f} cm from iso "
                  f"vs fantoma ±{max_phantom_half:.1f} cm. "
                  f"Geometric overlap expected (does not affect physics).")
    else:
        print(f"    Gantry:    0° (no rotation) | Source at (0, 0, {ssd:.1f}) cm")

    # =====================================================================
    # STEP 16: DOSEACTOR (dose detector — 2 mm resolution)
    # =====================================================================
    dose_full = sim.add_actor('DoseActor', 'dose_full')
    dose_full.attached_to = 'patient'
    dose_nx = int(phantom_size[0] * 10 / dose_spacing)
    dose_ny = int(phantom_size[1] * 10 / dose_spacing)
    dose_nz = int(phantom_size[2] * 10 / dose_spacing)
    dose_full.size = [dose_nx, dose_ny, dose_nz]
    dose_full.spacing = [dose_spacing * mm, dose_spacing * mm, dose_spacing * mm]
    dose_full.output_filename = str(out_path / f'{scenario_id}.mhd')
    dose_full.dose.active = True
    dose_full.dose_uncertainty.active = True
    # POST-KAGGLE FIX: disable edep to save disk space.
    # The deposited energy (edep) is not used for neural network training
    # nor for HDF5 dataset preparation. We only need dose and uncertainty.
    # Savings: ~64 MB/sim (2 × 200³ × 4 bytes), crucial on Kaggle (~20 GB).
    dose_full.edep.active = False
    dose_full.edep_uncertainty.active = False

    print(f"    DoseActor: {dose_nx}×{dose_ny}×{dose_nz} voxels | "
          f"Spacing {dose_spacing} mm | Attached to 'patient'")

    # =====================================================================
    # STEP 17: COMPLETE METADATA (includes TrueBeam + MLC info)
    # =====================================================================
    metadata = {
        'scenario_id': scenario_id,
        'mode': 'simplified_ebrt',
        'machine': 'Varian_TrueBeam_SVC_3.0',
        'mlc_model': 'HD_120_MLC',
        'patient_type': patient_type,
        'timestamp': datetime.now().isoformat(),
        'beam': {
            'particle': 'gamma',
            'beam_quality': beam_quality,
            'is_fff': is_fff,
            'emax_MeV': spectrum['emax_MeV'],
            'mean_energy_MeV': spectrum['mean_energy_MeV'],
            'dmax_cm': spectrum['dmax_cm'],
            'pdd_10cm': spectrum.get('pdd_10cm'),
            'd20_d10': spectrum.get('d20_d10'),
            'n_particles': n_particles,
            'gantry_angle_deg': gantry_angle,
            'collimator_angle_deg': collimator_angle,
            'ssd_cm': ssd,
        },
        'field': {
            'jaw_size_x_cm': field_x,
            'jaw_size_y_cm': field_y,
            'type': 'jaws_plus_HD120_MLC',
        },
        'mlc': {
            'n_pairs': MLC_N_PAIRS,
            'n_inner_pairs': MLC_N_INNER,
            'inner_width_mm': MLC_INNER_WIDTH_MM,
            'outer_width_mm': MLC_OUTER_WIDTH_MM,
            'thickness_cm': MLC_THICKNESS_CM,
            'sad_cm': MLC_SAD_CM,
            'n_open_pairs': n_mlc_open,
            'positions_cm': mlc_positions,
        },
        'geometry': {
            'phantom_size_cm': phantom_size,
            'tumor_depth_cm': tumor_depth,
            'tumor_size_cm': tumor_size,
            'tumor_material': tumor_mat,
            'bone_depth_cm': bone_depth,
            'bone_thickness_cm': bone_thick,
            'oar_position_cm': oar_pos,
            'oar_radius_cm': oar_radius,
            'has_lung': has_lung,
            'lung_position_cm': lung_pos if has_lung else None,
            'lung_size_cm': lung_size if has_lung else None,
        },
        'scoring': {'dose_spacing_mm': dose_spacing},
        'computation': {'random_seed': random_seed, 'n_threads': n_threads},
    }

    # Save JSON metadata
    meta_file = str(out_path / f'{scenario_id}_metadata.json')
    with open(meta_file, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)

    # =====================================================================
    # STEP 18: RUN THE MONTE CARLO SIMULATION
    # =====================================================================
    print(f"    ▶ Launching MC simulation...")
    try:
        sim.run()
    finally:
        try:
            sim.close()
        except Exception:
            pass

    print(f"  ✓ {scenario_id} completado")
    return metadata


# Confirm that the function has been defined correctly
if GATE_AVAILABLE:
    print("=" * 70)
    print("  Function run_ebrt_simplified() defined — Varian TrueBeam SVC 3.0")
    print("=" * 70)
    print(f"  Chain: Source → Jaws Y (z=SSD-55) → Jaws X (z=SSD-63) → HD 120 MLC (z=SSD-72) → Phantom")
    print(f"  MLC: {MLC_N_PAIRS} pairs ({MLC_N_INNER}×{MLC_INNER_WIDTH_MM}mm + "
          f"{MLC_N_OUTER_PER_SIDE*2}×{MLC_OUTER_WIDTH_MM}mm)")
    print(f"  Gap jaw_x → MLC: {63 - 72 + 7.8/2 + 6.5/2:.2f} cm (no overlap ✓)")
    print(f"  Qualities: 6MV, 6FFF, 10MV, 10FFF, 15MV")
    print(f"  Tipos: pelvis, head_neck, lung")
    print(f"  World: X and Z dynamic (ssd + 40 cm), supports all gantry angles")
    print(f"  DoseActor: dose + uncertainty (edep DISABLED to save disk)")
    print(f"  Overlap check: DISABLED (required for gantry ≠ 0° with SSD < 93 cm)")
else:
    print("=" * 70)
    print("  ⚠ OpenGATE NOT AVAILABLE — run_ebrt_simplified() defined but")
    print("  cannot be executed. Scenario generation functions,")
    print("  MLC, and analysis functions work normally.")
    print("=" * 70)

## 5 — Clinical scenario generator: 3 patient types × 5 beam qualities

To train a neural network that predicts 3D dose maps from the TrueBeam SVC 3.0, we need a **diverse dataset** covering clinically relevant combinations. The hospital will provide data from 3 patient types: **pelvis, head-and-neck, and lung**, treated with the 5 available beam qualities (6MV, 6FFF, 10MV, 10FFF, 15MV).

### Stratified sampling by patient type and beam quality

We distribute scenarios equitably among the 3 patient types and 5 beam qualities. However, not all combinations are clinically frequent — the generator applies realistic correlations:

| Patient type | Frequent beam qualities | Clinical justification |
|---|---|---|
| **Pelvis** | 10MV, 15MV, 10FFF | Deep tumors (prostate, cervix), need penetration |
| **Head and neck** | 6MV, 6FFF | Superficial/medium tumors, compact anatomy |
| **Lung** | 6MV, 6FFF, 10FFF | SBRT with FFF (high dose rate), small fields |

### Patient-specific anatomy

Each patient type generates a distinct anatomy:
- **Pelvis**: large phantom, thick bone at medium depth, deep tumor, no lung
- **Head and neck**: compact phantom, thin bone, superficial tumor, nearby OARs
- **Lung**: phantom with bilateral lung inserts, intra-pulmonary tumor, extreme heterogeneities

### HD 120 MLC in scenarios

The positions of all 120 MLC leaves are calculated analytically (circular conformation) at this stage. When DICOM data from the hospital becomes available, they will be replaced with real positions from the Eclipse v17.0.0 inverse planner.


In [ ]:
# =============================================================================
# CLINICAL EBRT SCENARIO GENERATOR — TRUEBEAM SVC 3.0 + HD 120 MLC
# =============================================================================
# CHANGES FROM THE PREVIOUS VERSION:
#   1. BEAM_QUALITIES: removed '18MV', added '6FFF' and '10FFF'
#   2. ADDED: PATIENT_TYPES = ['pelvis', 'head_neck', 'lung']
#   3. ADDED: Patient type ↔ beam quality correlation
#   4. ADDED: Patient-type-specific anatomy (phantom size,
#      materials, positions, lung presence)
#   5. ADDED: Analytical MLC positions for each scenario
#   6. UPDATED: Stratified sampling by (patient_type × beam_quality)
#
# POST-KAGGLE FIXES:
#   7. FIXED: All patient types use phantom_size = [40, 40, 40] cm.
#      Reason: head_neck used [20, 20, 25] → dose map shape (125, 100, 100)
#      instead of (200, 200, 200), causing the HDF5 builder to reject
#      2 of 5 scenarios ("incorrect shape"). Now ALL phantoms are
#      40×40×40 cm, ensuring uniform dose maps of (200, 200, 200).
#      Head-and-neck structures are placed inside this larger
#      phantom (the zone without structures is pure water = soft tissue).
# =============================================================================

import random
import itertools

# --- Beam qualities of the Varian TrueBeam SVC 3.0 ---
# 3 beams with flattening filter (FF) + 2 beams without filter (FFF)
# There is NO 18 MV on the TrueBeam SVC 3.0 (maximum FF energy = 15 MV)
BEAM_QUALITIES = ['6MV', '6FFF', '10MV', '10FFF', '15MV']

# --- Patient types (the 3 that the hospital will provide) ---
PATIENT_TYPES = ['pelvis', 'head_neck', 'lung']

# --- Clinical correlations: which beam qualities are used with each type ---
# These probabilities reflect real clinical practice:
#   - Pelvis: deep tumors → high energies (10, 15 MV) + 10FFF for fast VMAT
#   - Head-and-neck: compact anatomy → 6 MV/FFF (little penetration needed)
#   - Lung: SBRT with FFF (high dose rate) + 6/10 MV conventional
PATIENT_BEAM_WEIGHTS = {
    'pelvis':    {'6MV': 0.10, '6FFF': 0.05, '10MV': 0.30, '10FFF': 0.30, '15MV': 0.25},
    'head_neck': {'6MV': 0.40, '6FFF': 0.35, '10MV': 0.15, '10FFF': 0.05, '15MV': 0.05},
    'lung':      {'6MV': 0.15, '6FFF': 0.25, '10MV': 0.15, '10FFF': 0.30, '15MV': 0.15},
}

# --- Field sizes (jaws) ---
# 3×3 = SBRT-type fields (with HD MLC at 2.5 mm this is clinically relevant)
# 10×10 = dosimetric reference field
# 20×20 = large conventional fields (we do not exceed 22 cm due to MLC Y limit)
FIELD_SIZES = [(3, 3), (5, 5), (8, 8), (10, 10), (15, 15), (20, 20)]

# --- SSD --- (100 cm = isocentric standard)
SSDS = [80, 90, 100, 110]

# --- Gantry angles (IEC 61217) ---
GANTRY_ANGLES = [0, 45, 90, 135, 180, 270]

# --- Tumor depths (cm from surface) ---
TUMOR_DEPTHS = [2, 4, 6, 8, 10, 12, 15, 18]

# --- Tumor sizes (diameter, cm) ---
TUMOR_SIZES = [1, 2, 3, 4, 6, 8]

# --- Collimator angles ---
COLLIMATOR_ANGLES = [0, 45, 90]

# Minimum margin between structures to avoid GEANT4 overlap
_MARGIN = 0.1

# --- Uniform phantom size for ALL patient types ---
# POST-KAGGLE FIX: 40×40×40 cm is enforced to guarantee that
# ALL dose maps have the same shape (200×200×200 with 2mm spacing).
# The Kaggle log showed that head_neck with [20,20,25] produced shape
# (125,100,100), incompatible with the HDF5 dataset.
PHANTOM_SIZE_CM = [40.0, 40.0, 40.0]


def sample_anatomy_by_patient_type(patient_type, beam_quality, field_x, field_y):
    """
    Generates a realistic anatomy adapted to the patient type and beam quality.

    Each patient type has:
      - UNIFORM phantom of 40×40×40 cm (for consistent dose maps)
      - Distinct tumor depth range
      - Specific materials and densities
      - Presence/absence of lung
      - Contextualized bone position

    Returns: dict with all anatomical parameters of the scenario
    """
    spec = BREMSSTRAHLUNG_SPECTRA[beam_quality]
    d80 = spec['d80_cm']
    dmax = spec['dmax_cm']

    # All patient types use the same phantom size
    phantom_size = list(PHANTOM_SIZE_CM)

    if patient_type == 'pelvis':
        # ==================================================================
        # PELVIS: deep tumors, thick bone
        # ==================================================================
        # Pelvis patients (prostate, cervix, rectum, bladder) are treated
        # with medium-large fields and high energies (10-15 MV) because
        # tumors are at >10 cm depth, surrounded by pelvic bone.

        # Deep tumor (8-18 cm depending on beam quality)
        td_range = (max(6, int(dmax + 3)), min(18, int(d80 + 8)))
        tumor_depth = random.uniform(td_range[0], td_range[1])
        tumor_size = random.choice([3, 4, 6, 8])  # Medium-large tumors

        # Pelvic bone: thick (0.8-1.5 cm), at medium depth
        bone_depth = random.uniform(max(3.0, dmax + 1), min(15.0, tumor_depth - 2))
        bone_thick = random.choice([0.8, 1.0, 1.5])

        # OAR: bladder or femoral head (lateral, medium depth)
        oar_lateral = random.uniform(3.0, 10.0)
        oar_depth = random.uniform(max(5.0, dmax + 2), min(25.0, d80 + 8))
        oar_r = random.choice([1.5, 2.0, 2.5])

        has_lung = False
        lung_pos = None
        lung_size = None

    elif patient_type == 'head_neck':
        # ==================================================================
        # HEAD AND NECK: superficial tumors, many nearby OARs
        # ==================================================================
        # Head-and-neck anatomy is the most challenging for planning
        # because it has critical structures very close to each other (parotids,
        # spinal cord, optic chiasm, brainstem, lenses).
        # Tumors are usually superficial (2-8 cm) and treated with 6 MV.
        #
        # NOTE: Although the real head-and-neck anatomy is ~20×20×25 cm,
        # we use the standard 40×40×40 cm phantom to maintain
        # dose map consistency. The anatomical structures
        # are placed in the upper-central zone of the phantom. The zone without
        # structures is pure water (equivalent to soft tissue).

        # Superficial-medium tumor (2-8 cm)
        tumor_depth = random.uniform(2.0, 8.0)
        tumor_size = random.choice([1, 2, 3, 4])  # Small-medium tumors

        # Thin bone (skull base, mandible): 0.3-0.8 cm
        bone_depth = random.uniform(max(1.0, dmax), min(10.0, tumor_depth + 3))
        bone_thick = random.choice([0.3, 0.5, 0.8])

        # OAR: parotid, spinal cord → small, nearby structure
        oar_lateral = random.uniform(1.0, 6.0)
        oar_depth = random.uniform(max(2.0, dmax), min(15.0, d80 + 3))
        oar_r = random.choice([0.5, 0.8, 1.0, 1.5])

        has_lung = False
        lung_pos = None
        lung_size = None

    elif patient_type == 'lung':
        # ==================================================================
        # LUNG: extreme heterogeneity, CPE loss, SBRT
        # ==================================================================
        # Lung treatments are the most complex for MC because the
        # low density of lung tissue (0.26 g/cm³) causes loss
        # of lateral electronic equilibrium, penumbra broadening, and
        # sharp dose gradients at lung-tissue interfaces.

        # Tumor can be inside the lung (3-10 cm, varied)
        tumor_depth = random.uniform(3.0, 12.0)
        tumor_size = random.choice([1, 2, 3, 4])  # Typical pulmonary nodules

        # Chest wall (ribs): superficial thin bone
        bone_depth = random.uniform(1.0, max(2.0, dmax + 1))
        bone_thick = random.choice([0.3, 0.5, 0.8])

        # OAR: heart, esophagus, spinal cord
        oar_lateral = random.uniform(2.0, 10.0)
        oar_depth = random.uniform(max(3.0, dmax + 1), min(25.0, d80 + 5))
        oar_r = random.choice([1.0, 1.5, 2.0, 2.5])

        # Lung ALWAYS present in lung scenarios
        has_lung = True
        lung_depth = random.uniform(max(3.0, dmax + 1), min(20.0, d80 + 3))
        lung_lateral = random.uniform(-5.0, 5.0)
        lung_w = random.uniform(8.0, 14.0)
        lung_h = random.uniform(10.0, 16.0)
        lung_d = random.uniform(10.0, 18.0)
        lung_pos = [round(lung_lateral, 1), 0.0, round(lung_depth, 1)]
        lung_size = [round(lung_w, 1), round(lung_h, 1), round(lung_d, 1)]
    else:
        raise ValueError(f"Unrecognized patient type: {patient_type}")

    # Validate that the tumor fits inside the phantom
    phant_z = phantom_size[2]
    tumor_r = tumor_size / 2.0
    if tumor_depth + tumor_size > phant_z - 1.0:
        tumor_depth = phant_z - tumor_size - 1.0
    if tumor_depth < 1.0:
        tumor_depth = 1.0

    # Adjust bone if it extends outside the phantom
    if bone_depth + bone_thick > phant_z - 0.5:
        bone_depth = phant_z - bone_thick - 0.5

    # Adjust OAR
    if oar_depth - oar_r < 0.2:
        oar_depth = oar_r + 0.2
    if oar_depth + oar_r > phant_z - 0.5:
        oar_depth = phant_z - 0.5 - oar_r
    max_lat = phantom_size[0] / 2.0 - oar_r - 0.5
    if oar_lateral + oar_r > max_lat:
        oar_lateral = max_lat - oar_r

    return {
        'phantom_size': phantom_size,
        'tumor_depth': round(tumor_depth, 1),
        'tumor_size': tumor_size,
        'bone_depth': round(bone_depth, 1),
        'bone_thick': bone_thick,
        'oar_pos': [round(oar_lateral, 1), 0.0, round(oar_depth, 1)],
        'oar_r': oar_r,
        'has_lung': has_lung,
        'lung_pos': lung_pos,
        'lung_size': lung_size,
    }


def validate_ebrt_geometry(td, ts, bd, bt, oar_pos, oar_r,
                           has_lung, lung_pos, lung_size, phant_z=40.0):
    """
    Verifies that a scenario is geometrically consistent.
    GEANT4 does not allow overlap between sibling volumes.

    Checks: all structures within the phantom, and no
    overlaps between tumor-bone, tumor-OAR, OAR-bone, lung-all.
    """
    tumor_r = ts / 2.0
    tumor_top = td
    tumor_bot = td + ts
    tumor_center = td + tumor_r

    if tumor_top < 0.5 or tumor_bot > phant_z - 0.5:
        return False
    if bd + bt > phant_z - 0.5:
        return False
    if oar_pos[2] - oar_r < 0.2 or oar_pos[2] + oar_r > phant_z - 0.5:
        return False
    # Tumor vs Bone
    if (tumor_bot + _MARGIN) > bd and (tumor_top - _MARGIN) < (bd + bt):
        return False
    # Tumor vs OAR (3D distance)
    dist = (oar_pos[0]**2 + oar_pos[1]**2 + (oar_pos[2] - tumor_center)**2)**0.5
    if dist < tumor_r + oar_r + _MARGIN:
        return False
    # OAR vs Bone
    if (oar_pos[2] - oar_r - _MARGIN) < (bd + bt) and (oar_pos[2] + oar_r + _MARGIN) > bd:
        return False

    if has_lung and lung_pos is not None and lung_size is not None:
        lx, ly, lz = lung_pos
        lw, lh, ld = lung_size
        l_top = lz - ld / 2
        l_bot = lz + ld / 2
        if l_top < 0.5 or l_bot > phant_z - 0.5:
            return False
        closest_z = max(l_top, min(tumor_center, l_bot))
        closest_x = max(lx - lw/2, min(0, lx + lw/2))
        d_lung_tumor = ((closest_x)**2 + (closest_z - tumor_center)**2)**0.5
        if d_lung_tumor < tumor_r + _MARGIN:
            return False
        if (l_top - _MARGIN) < (bd + bt) and (l_bot + _MARGIN) > bd:
            return False

    return True


def _weighted_choice(weight_dict):
    """Weighted random selection from a dictionary {key: weight}."""
    keys = list(weight_dict.keys())
    weights = [weight_dict[k] for k in keys]
    return random.choices(keys, weights=weights, k=1)[0]


def generate_ebrt_scenarios(n_particles, n_threads, mode='random', n_random=20,
                            seed=42, prefix='E', g4_seed_offset=20000):
    """
    Generates EBRT scenarios for Varian TrueBeam SVC 3.0 with HD 120 MLC.

    Stratified sampling by patient type (pelvis, head_neck, lung).
    Within each type, beam quality is chosen with probabilities
    that reflect clinical practice (PATIENT_BEAM_WEIGHTS).

    Each scenario includes:
      - Beam parameters (quality, field, gantry, SSD)
      - Patient-type-specific anatomy
      - HD 120 MLC positions (analytical conformation)
      - Complete metadata for reproducibility

    Returns: list of dicts ready for run_ebrt_simplified()
    """
    if mode == 'random':
        random.seed(seed)
        scenarios = []

        # Distribute equally among the 3 patient types
        n_per_type = n_random // len(PATIENT_TYPES)
        remainder = n_random % len(PATIENT_TYPES)

        for ti, ptype in enumerate(PATIENT_TYPES):
            quota = n_per_type + (1 if ti < remainder else 0)
            att = 0

            while quota > 0 and att < quota * 500:
                att += 1

                # Choose beam quality with weighted probability by type
                quality = _weighted_choice(PATIENT_BEAM_WEIGHTS[ptype])

                # Sample field size
                fx, fy = random.choice(FIELD_SIZES)

                # MLC constraint: field Y cannot exceed 22 cm (HD 120 MLC limit)
                if fy > 22:
                    fy = 22

                ssd = random.choice(SSDS)
                gantry = random.choice(GANTRY_ANGLES)
                coll = random.choice(COLLIMATOR_ANGLES)

                # Generate patient-type-specific anatomy
                anat = sample_anatomy_by_patient_type(ptype, quality, fx, fy)

                td = anat['tumor_depth']
                ts = anat['tumor_size']

                # Constraint: the tumor must fit within the jaw field
                if ts > min(fx, fy) * 0.8:
                    continue

                phant_z = anat['phantom_size'][2]

                # Validate geometry
                if validate_ebrt_geometry(td, ts, anat['bone_depth'], anat['bone_thick'],
                                         anat['oar_pos'], anat['oar_r'],
                                         anat['has_lung'], anat['lung_pos'],
                                         anat['lung_size'], phant_z=phant_z):

                    # Calculate MLC positions to conform field to tumor
                    mlc_pos = compute_mlc_leaf_positions_from_field(
                        fx, fy, td, ts, ssd
                    )

                    scenarios.append({
                        'scenario_id': f'ebrt_{prefix}_{len(scenarios):04d}',
                        'beam_quality': quality,
                        'patient_type': ptype,
                        'field_size_x_cm': fx,
                        'field_size_y_cm': fy,
                        'ssd_cm': ssd,
                        'gantry_angle_deg': gantry,
                        'collimator_angle_deg': coll,
                        'tumor_depth_cm': td,
                        'tumor_size_cm': ts,
                        'tumor_material': 'G4_TISSUE_SOFT_ICRP',
                        'bone_depth_cm': anat['bone_depth'],
                        'bone_thickness_cm': anat['bone_thick'],
                        'oar_position_cm': anat['oar_pos'],
                        'oar_radius_cm': anat['oar_r'],
                        'has_lung': anat['has_lung'],
                        'lung_position_cm': anat['lung_pos'],
                        'lung_size_cm': anat['lung_size'],
                        'phantom_size_cm': anat['phantom_size'],
                        'mlc_positions': mlc_pos,
                        'dose_spacing_mm': 2.0,
                        'n_particles': n_particles,
                        'n_threads': n_threads,
                        'random_seed': g4_seed_offset + len(scenarios),
                    })
                    quota -= 1

        random.shuffle(scenarios)
        return scenarios
    else:
        raise NotImplementedError("Only random mode implemented for EBRT")


# --- Generator verification ---
test_scenarios = generate_ebrt_scenarios(100000, 4, mode='random', n_random=21, seed=42)
print(f"generate_ebrt_scenarios() — TrueBeam SVC 3.0 + HD 120 MLC")
print(f"  Scenarios generated: {len(test_scenarios)}")

# Distribution by patient type
from collections import Counter
type_dist = Counter(s['patient_type'] for s in test_scenarios)
print(f"  Distribution by type:     {dict(type_dist)}")

# Distribution by beam quality
q_dist = Counter(s['beam_quality'] for s in test_scenarios)
print(f"  Distribution by quality:  {dict(q_dist)}")

# Verify there is no 18MV (removed from TrueBeam SVC 3.0)
assert '18MV' not in q_dist, "ERROR: 18MV does not exist on the TrueBeam SVC 3.0"

# Verify uniform phantom sizes
phantom_sizes = set(tuple(s['phantom_size_cm']) for s in test_scenarios)
print(f"  Uniform phantom:   {list(phantom_sizes)} → all maps will be (200, 200, 200) ✓")

# Verify proportion of scenarios with lung
n_lung = sum(1 for s in test_scenarios if s.get('has_lung', False))
print(f"  With lung: {n_lung}/{len(test_scenarios)} ({100*n_lung/len(test_scenarios):.0f}%)")

# MLC stats
n_mlc_open = [sum(1 for pa, pb in s['mlc_positions'] if pb - pa > 0.1)
              for s in test_scenarios]
print(f"  MLC open leaves: min={min(n_mlc_open)}, max={max(n_mlc_open)}, "
      f"media={np.mean(n_mlc_open):.1f}")

## 6 — Production configuration: how many simulations and with what resources

Before launching the simulation batch, we must configure global parameters: how many scenarios to simulate, how many particles per simulation, how many CPU threads to use, and where to save results.

**Note on speed with HD 120 MLC**: Simulations with the 120 tungsten leaf model are ~25% slower than jaw-only simulations, because electromagnetic cascades in the MLC leaves generate additional secondaries that GEANT4 must track. The estimated throughput is ~150 photons/s/core (vs ~200 without MLC).

The fundamental trade-off is between **statistical precision** and **computational investment**. Statistical uncertainty decreases as $\sigma \propto 1/\sqrt{N}$, where $N$ is the number of simulated particles. With 1M photons and the HD 120 MLC modeled, we obtain uncertainties of ~3-5% in the in-field voxels.


In [ ]:
# =============================================================================
# PRODUCTION CONFIGURATION — VARIAN TRUEBEAM SVC 3.0 WITH HD 120 MLC
# =============================================================================
# This cell defines ALL parameters for the simulation batch.
# Adapted for the 5 TrueBeam beam qualities (6MV, 6FFF, 10MV, 10FFF, 15MV)
# and the 3 patient types (pelvis, head_neck, lung).
#
# OPTIMAL N_SCENARIOS CALCULATION PER NOTEBOOK:
#   - Kaggle: 12h CPU, ~10h effective (discounting install + init + HDF5)
#   - Average time per simulation (Kaggle log 5/5): 153.8s ≈ 160s
#   - High variability: 37s (small head_neck) → 355s (large pelvis)
#   - Calculation: 10h × 3600s / 160s ≈ 225 sims
#   - With safety margin (slow sims, timeout): 200 sims/notebook
#   - Auto-resume: if time runs out, relaunch to complete
# =============================================================================

import multiprocessing
import os

NOTEBOOK_ID = 70

# --- Execution mode ---
PRODUCTION_MODE = True

if PRODUCTION_MODE:
    N_SCENARIOS = 130      # ~200 sims × 160s = ~8.9h (within Kaggle's 12h)
    N_PARTICLES = 1000000  # 1M primary photons per simulation
else:
    N_SCENARIOS = 3        # Only 3 test scenarios
    N_PARTICLES = 200000   # 200K photons

SCENARIO_PREFIX = f'E{NOTEBOOK_ID:02d}'
GENERATOR_SEED = 41 + NOTEBOOK_ID
G4_SEED_OFFSET = 10000 * (NOTEBOOK_ID + 1)
N_THREADS = min(4, multiprocessing.cpu_count())
BATCH_OUTPUT = os.path.join(BASE_OUTPUT, 'ebrt_batch')

# =============================================================================
# RECOVER PREVIOUS SIMULATIONS (resume between Kaggle sessions)
# =============================================================================
# Kaggle deletes /kaggle/working/ on restart. To resume simulations:
#   1. Save the output of the previous notebook (Save & Run All → Output tab)
#   2. Create a new notebook and add that output as Input Dataset
#   3. Simulations will appear in /kaggle/input/<dataset-name>/ebrt_batch/
#   4. This block copies them to /kaggle/working/ebrt_batch/ so that
#      auto-resume detects and skips already completed ones.
# =============================================================================
if IN_KAGGLE:
    _prev_batch = None
    for _entry in os.listdir(BASE_INPUT):
        _candidate = os.path.join(BASE_INPUT, _entry, 'ebrt_batch')
        if os.path.isdir(_candidate):
            _prev_batch = _candidate
            break
    if _prev_batch:
        os.makedirs(BATCH_OUTPUT, exist_ok=True)
        _n_recovered = 0
        for _sim_dir in sorted(os.listdir(_prev_batch)):
            _src = os.path.join(_prev_batch, _sim_dir)
            _dst = os.path.join(BATCH_OUTPUT, _sim_dir)
            if os.path.isdir(_src) and not os.path.exists(_dst):
                shutil.copytree(_src, _dst)
                _n_recovered += 1
        # Copy batch_index.json if it exists
        _idx_src = os.path.join(_prev_batch, 'batch_index.json')
        _idx_dst = os.path.join(BATCH_OUTPUT, 'batch_index.json')
        if os.path.exists(_idx_src) and not os.path.exists(_idx_dst):
            shutil.copy2(_idx_src, _idx_dst)
        _free_after = shutil.disk_usage(BATCH_OUTPUT).free / 1e9
        print(f"  ✓ RESUME: {_n_recovered} simulations recovered from {_prev_batch}")
        print(f"    → The simulation loop will skip already completed ones")
        print(f"    → Disco libre tras copia: {_free_after:.1f} GB")
    else:
        print(f"  ℹ No previous simulations found in {BASE_INPUT}/*/ebrt_batch/")
        print(f"    First run or no input dataset added.")
else:
    os.makedirs(BATCH_OUTPUT, exist_ok=True)

# --- Time estimates ---
# Based on real Kaggle data (5/5 sims): mean 153.8s, range 37-355s.
# Rate varies greatly with complexity (open MLC, field size).
PHOTON_RATE = 150  # particles/second/core (with MLC → more cascades in W)

est_time_per_sim = N_PARTICLES / PHOTON_RATE / N_THREADS + 30  # +30s MLC initialization
est_total = est_time_per_sim * N_SCENARIOS

print(f"EBRT CONFIGURATION — TrueBeam SVC 3.0 + HD 120 MLC "
      f"({'PRODUCTION' if PRODUCTION_MODE else 'TEST'}):")
print(f"   Notebook:    #{NOTEBOOK_ID} (prefix '{SCENARIO_PREFIX}')")
print(f"   Scenarios:   {N_SCENARIOS} ({len(PATIENT_TYPES)} types × ~{N_SCENARIOS//len(PATIENT_TYPES)}/type)")
print(f"   Particles:   {N_PARTICLES:,}/simulation")
print(f"   Threads:     {N_THREADS}")
print(f"   Qualities:   {', '.join(BEAM_QUALITIES)}")
print(f"   Types:       {', '.join(PATIENT_TYPES)}")
print(f"   MLC:         HD 120 ({MLC_N_PAIRS} pairs, {MLC_N_INNER}×{MLC_INNER_WIDTH_MM}mm + "
      f"{MLC_N_OUTER_PER_SIDE*2}×{MLC_OUTER_WIDTH_MM}mm)")
print(f"   Output:      {BATCH_OUTPUT}/")
print(f"   Est. time:   ~{est_time_per_sim:.0f}s/sim × {N_SCENARIOS} = "
      f"~{est_total/60:.1f} min ({est_total/3600:.1f}h)")

# WARNING if estimated time exceeds Kaggle limit
if est_total > 10 * 3600:
    print(f"   ⚠ Estimated time ({est_total/3600:.1f}h) exceeds Kaggle's 10h effective limit.")
    print(f"     Auto-resume will complete the remaining ones in the next run.")

est_disk_mb = 65 * N_SCENARIOS  # ~65 MB/sim (dose + uncertainty, without edep)
print(f"   Est. disk:   ~{est_disk_mb/1000:.1f} GB (dose + uncertainty only)")

# --- Generate scenarios ---
scenarios = generate_ebrt_scenarios(
    N_PARTICLES, N_THREADS,
    mode='random', n_random=N_SCENARIOS,
    seed=GENERATOR_SEED,
    prefix=SCENARIO_PREFIX,
    g4_seed_offset=G4_SEED_OFFSET,
)

# --- Summary table ---
print(f"\n{len(scenarios)} scenarios generated:\n")
print(f"{'ID':<22} {'Tipo':<12} {'Quality':>8} {'Field':>10} {'SSD':>5} {'Gantry':>7} "
      f"{'Depth':>6} {'Size':>5} {'Lung':>5} {'MLC':>5}")
print("-" * 95)
for s in scenarios[:15]:
    lung_str = 'Yes' if s.get('has_lung') else 'No'
    n_open = sum(1 for pa, pb in s.get('mlc_positions', []) if pb - pa > 0.1)
    print(f"{s['scenario_id']:<22} {s['patient_type']:<12} {s['beam_quality']:>8} "
          f"{s['field_size_x_cm']}x{s['field_size_y_cm']:>3} {s['ssd_cm']:>5} "
          f"{s['gantry_angle_deg']:>6}° {s['tumor_depth_cm']:>6} "
          f"{s['tumor_size_cm']:>5} {lung_str:>5} {n_open:>5}")
if len(scenarios) > 15:
    print(f"... and {len(scenarios)-15} more")

## 7 — Simulation batch execution: multiprocessing and fault tolerance

Executing multiple Monte Carlo simulations requires a robust process management strategy. Each simulation runs in a **separate process** using Python's `multiprocessing` module (not multithreading). Why processes instead of threads?

First, GEANT4 manages its own internal global state (material tables, geometries, random number generators). If we tried to run two simulations with different geometries in the same process, GEANT4's internal tables would conflict. By using separate processes, each simulation has its own independent GEANT4 instance with its own memory space.

Second, `multiprocessing.Process` allows us to implement **timeouts**: if a simulation takes longer than 30 minutes (which could indicate a problem, such as an infinite loop in a degenerate geometry), we can terminate the process and continue with the next simulation without losing the entire batch.

Finally, the execution implements **automatic resume**: before launching each simulation, we check whether a previous result already exists for that scenario. If the dose and metadata files already exist (and contain no errors), we skip that simulation. This allows interrupting and resuming the batch without repeating work.


In [ ]:
# =============================================================================
# EBRT SIMULATION BATCH EXECUTION
# =============================================================================
# This block executes all simulations sequentially. Each one is
# launched in a separate process with timeout to avoid blocking.
# If execution is interrupted, it can be resumed: already
# completed simulations are automatically detected and skipped.
#
# POST-KAGGLE FIXES:
#   - More detailed prints: patient type, quality, field, gantry,
#     SSD, depth, tumor size, MLC stats, estimated time.
#   - Final summary with results table.
#   - GATE_AVAILABLE guard to skip simulations if OpenGATE not available.
#   - os.makedirs(BATCH_OUTPUT) before writing batch_index.json.
#   - Fixed SyntaxError in final summary block (mixed lines).
# =============================================================================

import time             # For measuring execution times
import json             # For saving/loading metadata
import os               # File system operations
import shutil           # For directory cleanup
import multiprocessing as mp  # Parallel processes


def _disk_free_gb(path='/kaggle/working'):
    """Returns free disk space (GB) on the partition of `path`."""
    try:
        st = os.statvfs(path)
        return (st.f_bavail * st.f_frsize) / (1024**3)
    except Exception:
        return float('inf')  # If it fails, do not block


def _cleanup_sim_files(out_dir, sid):
    """
    Removes unnecessary intermediate files after a successful simulation.

    KEPT:
      - {sid}_dose.mhd + {sid}_dose.raw    (dose map — needed for HDF5)
      - {sid}_dose_uncertainty.mhd + .raw   (uncertainty — optional but useful)
      - {sid}_metadata.json                 (metadata — needed for HDF5)

    REMOVED:
      - {sid}_edep*                         (never used, disabled in DoseActor)
      - Any other residual .mhd/.raw
    """
    keep_suffixes = {'_dose.mhd', '_dose.raw',
                     '_dose_uncertainty.mhd', '_dose_uncertainty.raw',
                     '_metadata.json'}
    keep_files = {f'{sid}{s}' for s in keep_suffixes}
    deleted_mb = 0
    for fname in os.listdir(out_dir):
        if fname not in keep_files and (fname.endswith('.raw') or fname.endswith('.mhd')):
            fpath = os.path.join(out_dir, fname)
            deleted_mb += os.path.getsize(fpath) / 1e6
            os.remove(fpath)
    return deleted_mb

# ============================================
# VERIFY THAT OPENGATE IS AVAILABLE
# ============================================
if not GATE_AVAILABLE:
    print("=" * 70)
    print("  ⚠ SIMULATIONS SKIPPED — OpenGATE not available")
    print("  The analysis, visualization, and HDF5 cells will continue")
    print("  working if data from previous simulations is available.")
    print("=" * 70)
    scenarios = []  # Empty so the loop does not iterate


def _worker_ebrt(params, output_dir):
    """
    Worker function that runs in a child process.

    Why a separate function?
    multiprocessing.Process requires the code to execute to be in a
    standalone function (cannot be an instance method or a
    complex closure). The child process is a COPY of the parent,
    with its own memory space where GEANT4 can run without
    interfering with the parent process.

    If the simulation fails (for any reason), we capture the exception
    and save it in the metadata file for later diagnosis.
    """
    sid = params.get('scenario_id', '?')
    meta_file = os.path.join(output_dir, f'{sid}_metadata.json')
    try:
        os.makedirs(output_dir, exist_ok=True)
        # Run the full simulation
        metadata = run_ebrt_simplified(params, output_dir=output_dir)
        # Save metadata (already saved inside run_ebrt_simplified,
        # but we overwrite here with the potentially updated version)
        with open(meta_file, 'w') as f:
            json.dump(metadata, f, indent=2)
    except Exception as e:
        # If something fails, save the error in the metadata file
        os.makedirs(output_dir, exist_ok=True)
        with open(meta_file, 'w') as f:
            json.dump({'scenario_id': sid, 'error': str(e)}, f, indent=2)


# List to accumulate results from all simulations
results = []
t0 = time.time()  # Start time of the entire batch

os.makedirs(BATCH_OUTPUT, exist_ok=True)

# Count already completed simulations (from resume or previous run)
_n_prev = 0
if os.path.isdir(BATCH_OUTPUT):
    for _d in os.listdir(BATCH_OUTPUT):
        _mf = os.path.join(BATCH_OUTPUT, _d, f'{_d}_metadata.json')
        _df = os.path.join(BATCH_OUTPUT, _d, f'{_d}_dose.mhd')
        if os.path.exists(_mf) and os.path.exists(_df):
            try:
                with open(_mf) as _f:
                    if 'error' not in json.load(_f):
                        _n_prev += 1
            except Exception:
                pass

print("=" * 70)
print(f"  START: {len(scenarios)} EBRT simulations — TrueBeam SVC 3.0")
print(f"  MLC: HD 120 | Overlap check: OFF | Phantom: 40×40×40 cm")
if _n_prev > 0:
    print(f"  RESUME: {_n_prev} previous simulations detected — skipping")
    print(f"  Pending: ~{len(scenarios) - _n_prev} new simulations")
print("=" * 70)

for i, params in enumerate(scenarios):
    sid = params['scenario_id']
    out_dir = os.path.join(BATCH_OUTPUT, sid)
    meta_file = os.path.join(out_dir, f'{sid}_metadata.json')

    # ============================================
    # VERIFY DISK SPACE BEFORE SIMULATING
    # ============================================
    free_gb = _disk_free_gb(BATCH_OUTPUT)
    if free_gb < 1.5:
        print(f"\n  ⚠ ESPACIO EN DISCO INSUFICIENTE: {free_gb:.1f} GB libres (< 1.5 GB)")
        print(f"    Stopping batch at [{i+1}/{len(scenarios)}] to avoid crash.")
        print(f"    Completed so far: {sum(1 for r in results if 'error' not in r)}")
        print(f"    → Relaunch the notebook to complete the remaining ones (auto-resume).")
        break

    # Scenario summary
    n_open = sum(1 for pa, pb in params.get('mlc_positions', []) if pb - pa > 0.1)
    lung_str = 'with lung' if params.get('has_lung') else 'no lung'
    print(f"\n  [{i+1}/{len(scenarios)}] {sid}")
    print(f"    {params['beam_quality']} | {params['patient_type']} | "
          f"Field {params['field_size_x_cm']}×{params['field_size_y_cm']} cm² | "
          f"MLC {n_open}/60 open")
    print(f"    Gantry {params['gantry_angle_deg']}° | SSD {params['ssd_cm']} cm | "
          f"Tumor Ø{params['tumor_size_cm']} cm a {params['tumor_depth_cm']} cm | "
          f"{lung_str}")

    # ============================================
    # RESUME: does this simulation already exist?
    # ============================================
    dose_exists = os.path.exists(os.path.join(out_dir, f'{sid}_dose.mhd'))
    if os.path.exists(meta_file) and dose_exists:
        with open(meta_file, 'r') as f:
            prev = json.load(f)
        if 'error' not in prev:
            print(f"    → Already completed previously — skipping")
            results.append(prev)
            continue

    # If there is a previous directory with error, clean it and retry
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)  # Delete directory with partial results
        print(f"    → Cleaning up previous results with errors")

    # ============================================
    # LAUNCH SIMULATION IN A SEPARATE PROCESS
    # ============================================
    print(f"    ▶ Launching simulation process...")
    t_sim = time.time()

    p = mp.Process(target=_worker_ebrt, args=(params, out_dir))
    p.start()

    # Wait with timeout of 30 minutes
    p.join(timeout=1800)
    elapsed = time.time() - t_sim

    # ============================================
    # RESULTS MANAGEMENT
    # ============================================
    if p.is_alive():
        p.terminate()
        p.join()
        results.append({'scenario_id': sid, 'error': 'Timeout (>30 min)'})
        print(f"    ✗ TIMEOUT tras {elapsed:.0f}s (>30 min) — terminando proceso")

    elif p.exitcode != 0 and not os.path.exists(meta_file):
        results.append({'scenario_id': sid, 'error': f'Exit code {p.exitcode}'})
        print(f"    ✗ Process failed with exit code {p.exitcode}")

    elif os.path.exists(meta_file):
        with open(meta_file, 'r') as f:
            result = json.load(f)
        if 'error' in result:
            print(f"    ✗ ERROR: {result['error'][:100]}")
        else:
            result.setdefault('computation', {})['elapsed_seconds'] = round(elapsed, 1)
            with open(meta_file, 'w') as f:
                json.dump(result, f, indent=2)

            # Verify that dose files exist and have reasonable size
            dose_file = os.path.join(out_dir, f'{sid}_dose.raw')
            dose_size_mb = os.path.getsize(dose_file) / 1e6 if os.path.exists(dose_file) else 0

            # Clean unnecessary files (edep, residuals) to save disk
            cleaned_mb = _cleanup_sim_files(out_dir, sid)
            cleanup_str = f" | Cleaned: {cleaned_mb:.0f} MB" if cleaned_mb > 0.1 else ""

            print(f"    ✓ Completado en {elapsed:.1f}s | "
                  f"Dose: {dose_size_mb:.1f} MB | "
                  f"Tasa: {params['n_particles']/elapsed:.0f} γ/s"
                  f"{cleanup_str}")
        results.append(result)
    else:
        results.append({'scenario_id': sid, 'error': 'No results'})
        print(f"    ✗ No results (no metadata or dose files)")

# ============================================
# FINAL BATCH SUMMARY
# ============================================
total_time = time.time() - t0
n_ok = sum(1 for r in results if 'error' not in r)
n_err = len(results) - n_ok

# Save a batch index for quick reference
os.makedirs(BATCH_OUTPUT, exist_ok=True)

index_file = os.path.join(BATCH_OUTPUT, 'batch_index.json')
with open(index_file, 'w') as f:
    json.dump({
        'n_total': len(scenarios), 'n_success': n_ok,
        'total_time_s': round(total_time, 1),
        'scenarios': [{'id': r.get('scenario_id', '?'),
                       'status': 'error' if 'error' in r else 'ok'} for r in results]
    }, f, indent=2)

print("\n" + "=" * 70)
print(f"  BATCH COMPLETED: {n_ok}/{len(scenarios)} successful"
      f"{f', {n_err} fallidas' if n_err > 0 else ''}")
print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
if n_ok > 0:
    times = [r.get('computation', {}).get('elapsed_seconds', 0)
             for r in results if 'error' not in r]
    print(f"  Avg time/sim: {np.mean(times):.1f}s | "
          f"Min: {min(times):.1f}s | Max: {max(times):.1f}s")
print("=" * 70)

## 8 — Loading and analyzing dose maps: from MHD files to NumPy arrays

Once the simulations have been executed, the results are stored in files with **MHD (MetaImage Header)** format. This format, originally developed for medical images, stores data in two separate files:

1. **`.mhd` file**: A plain text file containing the image metadata — the volume dimensions (number of voxels along each axis), the voxel spacing, the offset (position of the first voxel in world coordinates), and the data type (float32, float64, etc.). It also includes the name of the associated binary file.

2. **`.raw` or `.zraw` file**: A binary file containing the raw numerical values, one per voxel, stored sequentially with no header. The order is Z-major (voxels are traversed first by X, then by Y, then by Z).

To read these files, we implement two methods:

- **SimpleITK** (preferred): A Python library developed by the NIH (National Institutes of Health) specifically for medical image processing. It supports >20 different formats (DICOM, NIfTI, MHD, etc.) and automatically handles coordinate transformations. When available, we use SimpleITK because it is more robust.

- **Manual reading** (fallback): If SimpleITK is not installed, we parse the `.mhd` file line by line, extract the metadata, and load the binary file with `numpy.fromfile()`. This is more fragile but works without extra dependencies.


In [ ]:
# =============================================================================
# LOADING AND ANALYSIS OF EBRT DOSE MAPS
# =============================================================================
# This cell loads the 3D dose maps generated by OpenGATE and shows
# a detailed statistical summary. Dose maps are 3D tensors where
# each voxel contains the absorbed dose (in Gray) calculated by GEANT4.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt


def load_mhd_manual(mhd_path):
    """
    Loads an MHD (MetaImage Header) file manually, without SimpleITK.

    The .mhd file is plain text with "key = value" lines:
      ObjectType = Image
      NDims = 3
      DimSize = 200 200 200       (number of voxels in X, Y, Z)
      ElementSpacing = 2.0 2.0 2.0  (voxel size in mm)
      Offset = -199.0 -199.0 -199.0 (position of the first voxel in mm)
      ElementType = MET_FLOAT      (data type: float32)
      ElementDataFile = file.raw   (name of the binary file)

    The .raw file contains the binary data without a header:
      200 × 200 × 200 × 4 bytes = 32 MB of float32

    Returns: (3D array of shape [Nz, Ny, Nx], metadata dict)
    """
    # Parse the MHD header file
    info = {}
    with open(mhd_path, 'r') as f:
        for line in f:
            if '=' in line:
                k, v = line.split('=', 1)
                info[k.strip()] = v.strip()

    # Extract volume dimensions (number of voxels)
    dims = list(map(int, info['DimSize'].split()))       # [Nx, Ny, Nz]
    # Extract voxel spacing (in mm)
    spacing = list(map(float, info['ElementSpacing'].split()))  # [dx, dy, dz] mm
    # Extract offset (position of the first voxel center, in mm)
    offset = list(map(float, info.get('Offset', '0 0 0').split()))

    # Map MHD data type to the corresponding numpy type
    dtype_map = {
        'MET_FLOAT': np.float32,    # 4 bytes per voxel (most common)
        'MET_DOUBLE': np.float64,   # 8 bytes per voxel (double precision)
        'MET_SHORT': np.int16,      # 2 bytes (for CT images)
        'MET_UCHAR': np.uint8,      # 1 byte (for binary masks)
    }
    dtype = dtype_map.get(info['ElementType'], np.float32)

    # Load binary file (.raw) and reshape to 3D volume
    # Reshape order is [Nz, Ny, Nx] (Z-major, following the medical
    # imaging convention where Z is the CT couch axis)
    raw_file = os.path.join(os.path.dirname(mhd_path), info['ElementDataFile'])
    data = np.fromfile(raw_file, dtype=dtype).reshape(dims[2], dims[1], dims[0])
    return data, {'dims': dims, 'spacing': spacing, 'offset': offset}


def load_mhd(mhd_path):
    """
    Loads an MHD file using SimpleITK (preferred) or manual reading (fallback).

    SimpleITK is much more robust than manual reading because:
    - Supports zlib compression (.zraw files)
    - Automatically handles coordinate transformations
    - Correctly manages data types and byte order
    """
    try:
        import SimpleITK as sitk
        img = sitk.ReadImage(mhd_path)
        # GetArrayFromImage returns a numpy array in [Z, Y, X] order
        arr = sitk.GetArrayFromImage(img).astype(np.float64)
        meta = {
            'dims': list(img.GetSize()),       # [Nx, Ny, Nz]
            'spacing': list(img.GetSpacing()),  # [dx, dy, dz] mm
            'offset': list(img.GetOrigin()),    # [ox, oy, oz] mm
        }
        return arr, meta
    except ImportError:
        # SimpleITK not available → use manual reading
        return load_mhd_manual(mhd_path)


# === Find the first completed simulation ===
first_sim_dir = None
first_sim_meta = None
dose_3d = None
dose_meta = None
if os.path.exists(BATCH_OUTPUT):
    for d in sorted(os.listdir(BATCH_OUTPUT)):
        candidate = os.path.join(BATCH_OUTPUT, d)
        meta_candidate = os.path.join(candidate, f'{d}_metadata.json')
        if os.path.isdir(candidate) and os.path.exists(meta_candidate):
            with open(meta_candidate) as _f:
                _meta = json.load(_f)
            if 'error' not in _meta:
                first_sim_dir = candidate
                first_sim_meta = _meta
                break

if first_sim_dir is None:
    print("  ✗ No outputs found. Run the simulation cells first.")
else:
    sim_id = os.path.basename(first_sim_dir)
    print(f"{'='*60}")
    print(f"  SIMULATION ANALYSIS: {sim_id}")
    print(f"{'='*60}")

    # Scenario info
    if first_sim_meta:
        beam = first_sim_meta.get('beam', {})
        geom = first_sim_meta.get('geometry', {})
        mlc_info = first_sim_meta.get('mlc', {})
        print(f"  Type: {first_sim_meta.get('patient_type', '?')} | "
              f"Beam: {beam.get('beam_quality', '?')} | "
              f"Gantry: {beam.get('gantry_angle_deg', 0)}° | "
              f"SSD: {beam.get('ssd_cm', 100)} cm")
        print(f"  Tumor: Ø{geom.get('tumor_size_cm', '?')} cm at "
              f"{geom.get('tumor_depth_cm', '?')} cm | "
              f"MLC: {mlc_info.get('n_open_pairs', '?')}/{mlc_info.get('n_pairs', 60)} open")

    # List all files generated by the simulation
    print(f"\n  Archivos generados:")
    files = sorted(os.listdir(first_sim_dir))
    total_size = 0
    for f in files:
        fp = os.path.join(first_sim_dir, f)
        sz = os.path.getsize(fp)
        total_size += sz
        sz_str = f"{sz/1e6:.2f} MB" if sz > 1e6 else f"{sz/1e3:.1f} KB"
        print(f"    {f:<50} {sz_str:>10}")
    print(f"    {'TOTAL':<50} {total_size/1e6:.1f} MB")

    # Load the 3D dose map
    dose_file = os.path.join(first_sim_dir, f'{sim_id}_dose.mhd')
    dose_3d = None
    if os.path.exists(dose_file):
        dose_3d, dose_meta = load_mhd(dose_file)
        print(f"\n  3D dose map:")
        print(f"    Shape:   {dose_3d.shape}")
        print(f"    Spacing: {dose_meta['spacing']} mm")
        print(f"    Offset:  {dose_meta['offset']} mm")
        print(f"    Range:   [{dose_3d.min():.4e}, {dose_3d.max():.4e}] Gy")
        nonzero = dose_3d[dose_3d > 0]
        if len(nonzero) > 0:
            print(f"    Voxels with dose > 0: {len(nonzero):,} / {dose_3d.size:,} "
                  f"({100*len(nonzero)/dose_3d.size:.1f}%)")
            print(f"    Mean dose (>0):       {nonzero.mean():.4e} Gy")
            print(f"    Median dose (>0):     {np.median(nonzero):.4e} Gy")
        else:
            print(f"    ⚠ WARNING: No voxels with dose > 0!")

        # Load uncertainty if exists
        uncert_file = os.path.join(first_sim_dir, f'{sim_id}_dose_uncertainty.mhd')
        if os.path.exists(uncert_file):
            uncert_3d, _ = load_mhd(uncert_file)
            uncert_valid = uncert_3d[dose_3d > 0]
            if len(uncert_valid) > 0:
                print(f"    Mean uncertainty:      {uncert_valid.mean():.1%}")
                print(f"    Uncertainty < 5%:      "
                      f"{100*np.sum(uncert_valid < 0.05)/len(uncert_valid):.1f}% de voxels")

        print(f"\n  NOTE: The absolute dose ({dose_3d.max():.2e} Gy) is very low because")
        print(f"  this is an MC simulation without Monitor Unit (MU) calibration. Clinically,")
        print(f"  the dose is normalized to the prescribed dose (e.g., 2 Gy/fraction).")

## 9 — Dose distribution visualization: 2D slices and dosimetric profiles

3D dose maps are difficult to interpret directly (they are 200×200×200 tensors of numbers). To analyze them, we "slice" them along three orthogonal planes passing through the maximum dose point:

- **Axial slice** (XY plane at fixed depth): Shows the lateral dose distribution in a horizontal plane. This is the most commonly used slice in clinical planning. It reveals the field size, penumbra, and isodose lines.

- **Coronal slice** (XZ plane at fixed Y): Shows the depth distribution in a frontal cross-section. It clearly reveals the buildup phenomenon, exponential attenuation, and perturbations from heterogeneities (bone, lung).

- **Sagittal slice** (YZ plane at fixed X): Similar to the coronal view but in the lateral plane.

Over these slices, we draw **isodose lines** (contour curves) at clinically relevant percentages: 10%, 20%, 50%, 80%, 90%, 95%. In a real treatment plan, the 95% and 90% isodose lines should completely envelop the tumor, while the 50% and 20% isodose lines indicate which healthy tissues receive significant dose.

In addition to 2D slices, we generate three **one-dimensional dosimetric profiles**:

- **PDD (Percentage Depth Dose)**: Dose along the beam's central axis as a function of depth. This curve is the "fingerprint" of the beam quality: it identifies the beam energy, the depth of maximum, and the attenuation.

- **Cross-plane profile**: Dose along the X axis (perpendicular to the beam axis) at the depth of maximum. Shows field uniformity and penumbra width.

- **In-plane profile**: Similar, along the Y axis.


In [ ]:
# =============================================================================
# EBRT DOSE DISTRIBUTION VISUALIZATION
# =============================================================================
# Generates 6 subplots: 3 2D slices with isodose + PDD + 2 lateral profiles.
# These plots are the standard way to present dosimetric results.
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np

if dose_3d is not None and dose_3d.max() > 0:
    nz, ny, nx = dose_3d.shape

    # Maximum absolute dose (in Gy), used as reference for normalization
    dose_max = dose_3d.max()

    # Position of the voxel with maximum dose (the "hot spot")
    # np.unravel_index converts the linear index to 3D index [z, y, x]
    max_idx = np.unravel_index(dose_3d.argmax(), dose_3d.shape)
    z_max, y_max, x_max = max_idx

    # Create the figure with 2×3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Normalize dose to percentage of maximum (0-100%)
    # This is the standard convention in clinical dosimetry
    dose_norm = dose_3d / dose_max * 100

    # --- 2D SLICES ---
    # We extract the three orthogonal planes passing through the max dose point
    slices_info = [
        # (2D array, title, X axis label, Y axis label)
        (dose_norm[z_max, :, :], f'Axial (Z={z_max})', 'X', 'Y'),
        (dose_norm[:, y_max, :], f'Coronal (Y={y_max})', 'X', 'Z'),
        (dose_norm[:, :, x_max], f'Sagital (X={x_max})', 'Y', 'Z'),
    ]

    # Clinically relevant isodose levels:
    # 95% = PTV coverage (Planning Target Volume)
    # 90% = minimum acceptable tolerance
    # 80% = d80%, defines the "useful depth" of the beam
    # 50% = reference for defining field size
    # 20% = low dose but still biologically significant
    # 10% = residual dose
    isodose_levels = [10, 20, 50, 80, 90, 95]
    isodose_colors = ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']

    for i, (sl, title, xlabel, ylabel) in enumerate(slices_info):
        ax = axes[0, i]
        # 2D image colored by dose (colormap "hot": black→red→yellow→white)
        im = ax.imshow(sl, cmap='hot', vmin=0, vmax=100,
                       aspect='auto', interpolation='bilinear')
        # Overlay isodose curves as contours
        for level, color in zip(isodose_levels, isodose_colors):
            ax.contour(sl, levels=[level], colors=[color], linewidths=0.8, alpha=0.8)
        ax.set_title(f'{title} — % max dose', fontsize=11, fontweight='bold')
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        plt.colorbar(im, ax=ax, label='%', shrink=0.8)

    # --- PDD (Percentage Depth Dose) ---
    # We extract the dose along the beam central axis (x=x_max, y=y_max)
    # as a function of Z (depth)
    sp = dose_meta['spacing']  # [dx, dy, dz] en mm
    pdd = dose_norm[:, y_max, x_max]  # 1D array of normalized dose

    # Convert indices to depth in cm
    offset_z = dose_meta['offset'][2]  # Z position of the first voxel (mm)
    phantom_hz = 20.0  # Half length of the phantom (40/2 = 20 cm)
    z_mm = np.arange(len(pdd)) * sp[2] + offset_z  # Z position of each voxel (mm)
    depth_cm = (phantom_hz - z_mm / 10)  # Convert to depth from the surface

    axes[1, 0].plot(depth_cm, pdd, 'b-', linewidth=2)
    axes[1, 0].set_xlabel('Depth (cm)', fontsize=11)
    axes[1, 0].set_ylabel('% Maximum dose', fontsize=11)
    axes[1, 0].set_title('PDD — Photons', fontsize=11, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    # Reference lines: 50% and 80% are the key clinical depths
    axes[1, 0].axhline(y=50, color='r', linestyle='--', alpha=0.5, label='50%')
    axes[1, 0].axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='80%')
    axes[1, 0].invert_xaxis()  # Depth increases to the right (convention)
    axes[1, 0].legend()

    # --- CROSS-PLANE PROFILE (X direction) ---
    # Dose along X at the depth of maximum (z=z_max, y=y_max)
    x_prof = dose_norm[z_max, y_max, :]
    x_mm = np.arange(len(x_prof)) * sp[0] + dose_meta['offset'][0]
    axes[1, 1].plot(x_mm, x_prof, 'r-', linewidth=2)
    axes[1, 1].set_xlabel('X (mm)', fontsize=11)
    axes[1, 1].set_ylabel('% Dose', fontsize=11)
    axes[1, 1].set_title('Perfil Cross-plane', fontsize=11, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

    # --- IN-PLANE PROFILE (Y direction) ---
    y_prof = dose_norm[z_max, :, x_max]
    y_mm = np.arange(len(y_prof)) * sp[1] + dose_meta['offset'][1]
    axes[1, 2].plot(y_mm, y_prof, 'g-', linewidth=2)
    axes[1, 2].set_xlabel('Y (mm)', fontsize=11)
    axes[1, 2].set_ylabel('% Dose', fontsize=11)
    axes[1, 2].set_title('Perfil In-plane', fontsize=11, fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)

    # Overall title and save
    fig.suptitle('Dose Distribution — EBRT with Photons',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_OUTPUT, 'ebrt_dose_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print("\nINTERPRETATION (photons):")
    print("  - The PDD shows buildup (dose increases up to dmax) + exponential attenuation")
    print("  - The lateral profiles show penumbra at the field edges (jaws)")
    print("  - The dose is NOT maximum at the surface (unlike electrons)")
else:
    print("No dose data to visualize")

## 10 — Dose-Volume Histogram (DVH): the fundamental clinical tool

The **DVH (Dose-Volume Histogram)** is arguably the most important plan evaluation tool in modern radiation therapy. A DVH shows, for each anatomical structure (tumor, OAR, healthy tissue), what percentage of its volume receives at least a given dose.

Mathematically, the cumulative DVH is defined as:

$$V(D) = \frac{1}{V_{total}} \int_D^{D_{max}} v(d) \, dd$$

where $v(d)$ is the differential volume receiving exactly dose $d$, and $V(D)$ is the fraction of the total volume receiving a dose equal to or greater than $D$. In practice, it is computed discretely by counting voxels: $V(D) = (\text{number of voxels with dose} \geq D) / (\text{total voxels in the structure})$.

The DVH is read from left to right: for the **tumor**, we want the curve to be as rectangular as possible (indicating that virtually the entire volume receives the same high dose), while for **OARs**, we want the curve to drop rapidly to zero (indicating that most of the organ receives little or no dose).

### Quality indices extracted from the DVH

- **D₉₅** (dose received by the most irradiated 95% of the volume): Used as an indicator of "minimum tumor coverage". In clinical practice, the acceptance criterion is D₉₅ ≥ 95% of the prescribed dose.

- **D₅** (dose received by the most irradiated 5% of the volume): An indicator of "hot spots". If D₅ greatly exceeds the prescribed dose, there are overdosed regions.

- **Homogeneity Index (HI)**: Defined as $HI = (D_5 - D_{95}) / D_{mean}$. Indicates how uniform the dose distribution is within the tumor. A perfect plan would have HI = 0 (perfectly uniform dose). In practice, HI < 0.1 is excellent, 0.1-0.2 is acceptable, > 0.2 needs improvement.

- **D_mean**: Mean dose in the structure. For OARs, lower is better. For the tumor, it should be close to the prescribed dose.

- **D_max**: Maximum dose (hot spot). It should never exceed 107% of the prescribed dose (ICRU 83 recommendation).


In [ ]:
# =============================================================================
# DOSE-VOLUME HISTOGRAM (DVH) — EBRT
# =============================================================================
# Calculates and plots the DVH for each anatomical structure of the simulation.
# Also calculates plan quality indices (HI, D95, D5, Dmean).
#
# The DVH is the standard way to evaluate a treatment plan in the clinic:
# medical physicists and oncologists use it in every planning session
# to decide if a plan is acceptable before irradiating the patient.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import json

if first_sim_dir and dose_3d is not None:
    # Load the metadata of the first simulation to determine the positions
    # of the anatomical structures
    meta_path = None
    for f in os.listdir(first_sim_dir):
        if f.endswith('_metadata.json'):
            meta_path = os.path.join(first_sim_dir, f)
            break

    if meta_path:
        with open(meta_path) as f:
            meta = json.load(f)

        # Dose map spacing (mm)
        sp = dose_meta['spacing']

        # Geometric parameters of the simulation
        geom = meta['geometry']
        phantom_hz = geom['phantom_size_cm'][2] / 2.0  # Half the phantom (20 cm)
        tumor_depth = geom['tumor_depth_cm']
        tumor_r = geom['tumor_size_cm'] / 2.0
        bone_depth = geom['bone_depth_cm']
        bone_thick = geom['bone_thickness_cm']
        oar_pos = geom['oar_position_cm']
        oar_r = geom['oar_radius_cm']

        nz, ny, nx = dose_3d.shape
        voxel_vol_mm3 = sp[0] * sp[1] * sp[2]
        voxel_vol_cm3 = voxel_vol_mm3 / 1e3

        print(f"{'='*60}")
        print(f"  DVH — DOSE ANALYSIS BY STRUCTURE")
        print(f"{'='*60}")
        print(f"  Dose grid: {nz}×{ny}×{nx} voxels")
        print(f"  Voxel: {sp[0]:.1f}×{sp[1]:.1f}×{sp[2]:.1f} mm³ = {voxel_vol_mm3:.1f} mm³")

        # ============================================
        # CREATE 3D MASKS for each structure
        # ============================================
        z_grid = np.arange(nz) * sp[2] + dose_meta['offset'][2]
        y_grid = np.arange(ny) * sp[1] + dose_meta['offset'][1]
        x_grid = np.arange(nx) * sp[0] + dose_meta['offset'][0]
        Z, Y, X = np.meshgrid(z_grid, y_grid, x_grid, indexing='ij')

        # --- TUMOR mask ---
        tumor_z_mm = (phantom_hz - tumor_depth - tumor_r) * 10
        tumor_mask = np.sqrt(X**2 + Y**2 + (Z - tumor_z_mm)**2) <= (tumor_r * 10)

        # --- BONE mask ---
        bone_z_mm = (phantom_hz - bone_depth - bone_thick / 2) * 10
        bone_mask = np.abs(Z - bone_z_mm) <= (bone_thick * 10 / 2)

        # --- OAR mask ---
        oar_z_mm = (phantom_hz - oar_pos[2]) * 10
        oar_mask = np.sqrt((X - oar_pos[0]*10)**2 + (Y - oar_pos[1]*10)**2 +
                          (Z - oar_z_mm)**2) <= (oar_r * 10)

        # --- LUNG mask (if exists) ---
        lung_mask = np.zeros_like(dose_3d, dtype=bool)
        if geom.get('has_lung') and geom.get('lung_position_cm'):
            lp = geom['lung_position_cm']
            ls = geom['lung_size_cm']
            lung_z = (phantom_hz - lp[2]) * 10
            lung_mask = (np.abs(X - lp[0]*10) <= ls[0]*10/2) & \
                        (np.abs(Y - lp[1]*10) <= ls[1]*10/2) & \
                        (np.abs(Z - lung_z) <= ls[2]*10/2)

        # --- HEALTHY TISSUE mask ---
        healthy_mask = (~tumor_mask) & (~bone_mask) & (~oar_mask) & \
                       (~lung_mask) & (dose_3d > 0)

        # Volume summary
        print(f"\n  Structure volumes:")
        print(f"    Tumor:       {np.sum(tumor_mask):>8,} voxels = "
              f"{np.sum(tumor_mask)*voxel_vol_cm3:>8.1f} cm³  "
              f"(r={tumor_r:.1f} cm, depth={tumor_depth:.1f} cm)")
        print(f"    Bone:        {np.sum(bone_mask):>8,} voxels = "
              f"{np.sum(bone_mask)*voxel_vol_cm3:>8.1f} cm³  "
              f"(depth={bone_depth:.1f} cm, thickness={bone_thick:.1f} cm)")
        print(f"    OAR:         {np.sum(oar_mask):>8,} voxels = "
              f"{np.sum(oar_mask)*voxel_vol_cm3:>8.1f} cm³  "
              f"(r={oar_r:.1f} cm, pos={oar_pos})")
        if lung_mask.any():
            print(f"    Lung:        {np.sum(lung_mask):>8,} voxels = "
                  f"{np.sum(lung_mask)*voxel_vol_cm3:>8.1f} cm³")
        print(f"    Healthy:     {np.sum(healthy_mask):>8,} voxels = "
              f"{np.sum(healthy_mask)*voxel_vol_cm3:>8.1f} cm³  (with dose > 0)")

        def compute_dvh(dose_array, mask, n_bins=200):
            """Calculates the cumulative DVH for a structure."""
            doses = dose_array[mask]
            if len(doses) == 0:
                return np.array([0]), np.array([0])
            d_max = doses.max()
            if d_max == 0:
                return np.array([0]), np.array([100])
            bins = np.linspace(0, d_max, n_bins)
            dvh = np.array([100 * np.sum(doses >= d) / len(doses) for d in bins])
            return bins, dvh

        # ============================================
        # GENERATE THE DVH PLOT
        # ============================================
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))

        structures = [
            ('Tumor (PTV)', tumor_mask, 'red', '-'),
            ('Bone', bone_mask, 'goldenrod', '--'),
            ('OAR', oar_mask, 'green', '--'),
            ('Healthy tissue', healthy_mask, 'blue', ':'),
        ]
        if lung_mask.any():
            structures.append(('Lung', lung_mask, 'cyan', '-.'))

        print(f"\n  Dosimetric statistics by structure:")
        print(f"  {'Structure':<20} {'Voxels':>8} {'D_mean (Gy)':>14} "
              f"{'D_max (Gy)':>14} {'D_mean/D_max':>12}")
        print(f"  {'-'*68}")

        for name, mask, color, ls in structures:
            doses = dose_3d[mask]
            if len(doses) > 0 and doses.max() > 0:
                bins, dvh = compute_dvh(dose_3d, mask)
                ax.plot(bins * 1e6, dvh, color=color, linestyle=ls,
                        linewidth=2, label=name)
                ratio = doses.mean() / doses.max() if doses.max() > 0 else 0
                print(f"  {name:<20} {np.sum(mask):>8,} {doses.mean():>14.4e} "
                      f"{doses.max():>14.4e} {ratio:>12.3f}")
            else:
                print(f"  {name:<20} {np.sum(mask):>8,} {'(no dose)':>14} "
                      f"{'—':>14} {'—':>12}")

        # ============================================
        # PLAN QUALITY INDICES (TUMOR)
        # ============================================
        tumor_doses = dose_3d[tumor_mask]
        if len(tumor_doses) > 0 and tumor_doses.max() > 0:
            d_mean = tumor_doses.mean()
            d_max_t = tumor_doses.max()
            d95 = np.percentile(tumor_doses[tumor_doses > 0], 5) \
                  if np.sum(tumor_doses > 0) > 0 else 0
            d5 = np.percentile(tumor_doses[tumor_doses > 0], 95) \
                 if np.sum(tumor_doses > 0) > 0 else 0
            HI = (d5 - d95) / d_mean if d_mean > 0 else 0

            print(f"\n  {'='*60}")
            print(f"  QUALITY INDICES (tumor):")
            print(f"    D₉₅  = {d95:.4e} Gy   (dose to the 95% most irradiated)")
            print(f"    D₅   = {d5:.4e} Gy   (dose to the 5% most irradiated)")
            print(f"    Dmean = {d_mean:.4e} Gy")
            print(f"    Dmax  = {d_max_t:.4e} Gy")
            print(f"    HI = (D₅ − D₉₅) / Dmean = {HI:.3f}")
            if HI < 0.1:
                print(f"    → Excellent homogeneity (HI < 0.1)")
            elif HI < 0.2:
                print(f"    → Good homogeneity (0.1 ≤ HI < 0.2)")
            elif HI < 0.5:
                print(f"    → Improvable homogeneity (0.2 ≤ HI < 0.5)")
            else:
                print(f"    → Poor homogeneity (HI ≥ 0.5). This is"
                      f" expected with a single beam + few MC particles.")
            print(f"  {'='*60}")

        ax.set_xlabel('Dose (μGy)', fontsize=12)
        ax.set_ylabel('Volumen (%)', fontsize=12)
        ax.set_title('DVH — EBRT with Photons', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([-2, 105])
        plt.tight_layout()
        plt.savefig(os.path.join(BASE_OUTPUT, 'ebrt_dvh.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No simulation data for DVH")

## 11 — Analytic density maps: the neural network input

To train a neural network that predicts dose maps, we need to provide it with two types of input information:

1. **Beam parameters** (numerical vector): beam quality, field size, SSD, gantry angle, etc.
2. **3D density map** (volumetric tensor): the spatial distribution of "patient" densities.

The density map is essentially a simplified version of what in clinical practice is obtained from a patient's **CT (computed tomography)**. A CT provides Hounsfield Unit (HU) values for each voxel, which are converted to relative electron densities via a calibration curve (CT-to-ED). In our case, since we use an analytic phantom, the density map is constructed directly from the positions and materials of the structures.

Each voxel in the density map contains a physical density value (g/cm³):
- **1.00**: water/soft tissue (includes the tumor, which is nearly indistinguishable by density)
- **1.85**: cortical bone (high Z → greater attenuation)
- **1.04**: brain tissue (OAR)
- **0.26**: lung (low density → less attenuation, longer electron range)

The neural network will use this density map together with the beam parameters to predict the 3D dose map. Conceptually, it is a function $f: (\rho_{3D}, \vec{p}) \rightarrow D_{3D}$, where $\rho_{3D}$ is the density tensor, $\vec{p}$ the parameter vector, and $D_{3D}$ the predicted dose tensor.


In [ ]:
# =============================================================================
# ANALYTICAL DENSITY MAP GENERATION FOR EBRT
# =============================================================================
# This function builds a 3D density map that exactly reproduces the
# MC simulation geometry. The density map is the main input
# of the neural network (along with the beam parameters).
#
# Why "analytical" and not from CT?
# Because our simulations use analytical phantoms (geometric shapes),
# not real CTs. The function reconstructs the same geometry (sphere for tumor,
# slab for bone, etc.) but as a numpy array of densities instead of
# a GEANT4 volume hierarchy.
# =============================================================================

import numpy as np


def build_ebrt_density(sim_meta, vol_shape, spacing_mm, offset_mm):
    """
    Builds an analytical 3D density map for an EBRT scenario.

    The map has the same resolution and position as the GEANT4 dose map,
    so there is a perfect voxel-to-voxel correspondence.

    Parameters:
        sim_meta: simulation metadata dict (with 'geometry')
        vol_shape: (Nz, Ny, Nx) — volume dimensions in voxels
        spacing_mm: [dx, dy, dz] — voxel size in mm
        offset_mm: [ox, oy, oz] — position of the first voxel in mm

    Returns:
        Numpy float32 array of shape vol_shape with densities in g/cm³
    """
    nz, ny, nx = vol_shape

    # Generate the coordinates (in mm) of each voxel center
    # These 1D arrays are expanded to a 3D mesh with meshgrid
    x = np.arange(nx, dtype=np.float32) * spacing_mm[0] + offset_mm[0]
    y = np.arange(ny, dtype=np.float32) * spacing_mm[1] + offset_mm[1]
    z = np.arange(nz, dtype=np.float32) * spacing_mm[2] + offset_mm[2]
    Z, Y, X = np.meshgrid(z, y, x, indexing='ij')

    # Initialize everything as water (density 1.0 g/cm³)
    # This is the base material of the phantom
    density = np.ones(vol_shape, dtype=np.float32)

    # Extract geometric parameters
    geom = sim_meta.get('geometry', {})
    phantom_hz = geom.get('phantom_size_cm', [40.0, 40.0, 40.0])[2] / 2.0

    # --- BONE (density 1.85 g/cm³) ---
    # Bone is modeled as a flat horizontal slab
    bd = geom.get('bone_depth_cm', 8.0)
    bt = geom.get('bone_thickness_cm', 0.5)
    bone_z = (phantom_hz - bd - bt / 2.0) * 10.0  # Bone center in mm
    bone_mask = np.abs(Z - bone_z) <= (bt * 10.0 / 2.0)
    density[bone_mask] = 1.85

    # --- TUMOR ---
    # The tumor has density ~1.00 (soft tissue), so we do not modify
    # the base water density. This is realistic: soft tumors are
    # practically invisible in the density map.

    # --- OAR (density 1.04 g/cm³) ---
    # The OAR (organ at risk) is a sphere of brain tissue
    oar_pos = geom.get('oar_position_cm', [5.0, 0.0, 8.0])
    oar_r = geom.get('oar_radius_cm', 1.5) * 10.0  # cm → mm
    oar_z = (phantom_hz - oar_pos[2]) * 10.0
    oar_mask = np.sqrt((X - oar_pos[0]*10)**2 + (Y - oar_pos[1]*10)**2 +
                       (Z - oar_z)**2) <= oar_r
    density[oar_mask] = 1.04

    # --- LUNG (density 0.26 g/cm³) ---
    # The lung is a low-density rectangular box
    if geom.get('has_lung') and geom.get('lung_position_cm'):
        lp = geom['lung_position_cm']
        ls = geom['lung_size_cm']
        lung_z = (phantom_hz - lp[2]) * 10.0
        lung_mask = (np.abs(X - lp[0]*10) <= ls[0]*10/2) & \
                    (np.abs(Y - lp[1]*10) <= ls[1]*10/2) & \
                    (np.abs(Z - lung_z) <= ls[2]*10/2)
        density[lung_mask] = 0.26

    return density


# --- Function verification ---
_test = build_ebrt_density(
    {'geometry': {
        'phantom_size_cm': [40, 40, 40], 'tumor_depth_cm': 5, 'tumor_size_cm': 3,
        'bone_depth_cm': 8, 'bone_thickness_cm': 0.5,
        'oar_position_cm': [5, 0, 10], 'oar_radius_cm': 1.5,
        'has_lung': True, 'lung_position_cm': [-6, 0, 15], 'lung_size_cm': [8, 10, 12],
    }},
    (200, 200, 200), [2.0, 2.0, 2.0], [-199.0, -199.0, -199.0]
)
print("build_ebrt_density() verified")
print(f"  Water (1.00): {(np.isclose(_test, 1.0)).sum():,} voxels")
print(f"  OAR  (1.04): {(np.isclose(_test, 1.04)).sum():,} voxels")
print(f"  Bone (1.85): {(np.isclose(_test, 1.85)).sum():,} voxels")
print(f"  Pulm (0.26): {(np.isclose(_test, 0.26)).sum():,} voxels")

## 12 — HDF5 dataset for the neural network: from Monte Carlo to deep learning

The final step of this pipeline is to compile ALL Monte Carlo simulation results into a single **HDF5** (Hierarchical Data Format version 5) file that will serve as the training dataset for the neural network.

HDF5 is a file format designed to efficiently store large volumes of numerical data. It functions like a "file system within a file": it contains **datasets** (multidimensional arrays) organized into **groups** (like directories), with support for transparent compression and random access. It is the standard format for deep learning datasets in medical imaging.

### Dataset structure

Our HDF5 file will contain 4 main datasets:

| Dataset | Shape | Type | Content |
|---|---|---|---|
| `/density` | `(N, 1, Dz, Dy, Dx)` | float32 | Density maps (INPUT for the network) |
| `/params` | `(N, P)` | float32 | Normalized beam parameters (INPUT) |
| `/dose` | `(N, 1, Dz, Dy, Dx)` | float32 | Dose maps (TARGET / Ground truth) |
| `/uncertainty` | `(N, 1, Dz, Dy, Dx)` | float32 | Statistical uncertainty |

where $N$ is the number of scenarios, $(Dz, Dy, Dx)$ the volume dimensions (200×200×200), and $P = 10$ the number of beam parameters.

### Beam parameter normalization (10 parameters)

The 10 beam parameters are normalized to the range [0, 1] to facilitate network training. They are **identical** to those expected by the neural network (`red_neuronal_EBRT.ipynb`, `Config.n_beam_params = 10`):

| # | Parameter | Real range | $p_{min}$ | $p_{max}$ | Notes |
|---|---|---|---|---|---|
| 0 | `beam_energy_norm` | 6/10/15 MV | 0.0 | 1.0 | 0.0=6MV, 0.5=10MV, 1.0=15MV |
| 1 | `is_fff` | 0/1 | 0 | 1 | 0=FF (with filter), 1=FFF (filter-free) |
| 2 | `field_x_cm` | 1–40 cm | 1 | 40 | Defined by X jaws |
| 3 | `field_y_cm` | 1–22 cm | 1 | 22 | HD MLC maximum: 22 cm |
| 4 | `ssd_cm` | 80–110 cm | 80 | 110 | Source-to-surface distance |
| 5 | `gantry_angle` | 0–360° | 0 | 360 | Gantry arm rotation |
| 6 | `collimator_angle` | 0–360° | 0 | 360 | MLC/field rotation |
| 7 | `patient_type` | pelvis/head/lung | 0.0 | 1.0 | 0=pelvis, 0.5=head-and-neck, 1.0=lung |
| 8 | `tumor_depth` | 2–18 cm | 2 | 18 | Distance from surface |
| 9 | `tumor_size` | 1–8 cm | 1 | 8 | Approximate diameter |

Density maps are normalized by dividing by 1.85 g/cm³ (compact bone density), so that bone=1.0, water=0.54, lung=0.14 — **consistent** with `generate_synthetic_phantom()` in the neural network. Dose maps are saved in Gy (absolute dose); per-sample normalization (÷ maximum_dose) is applied in the neural network during loading.

### Why is the channel dimension 1?

The shape `(N, 1, Dz, Dy, Dx)` follows the PyTorch/TensorFlow convention for 3D volumes: the second dimension is the **channel** (like the 3 RGB channels of an image). In our case, we only have 1 channel (density or dose), but we keep this dimension for compatibility with standard 3D convolutional network architectures (such as 3D U-Net, V-Net, etc.).


In [ ]:
# =============================================================================
# HDF5 DATASET PREPARATION FOR THE NEURAL NETWORK
# =============================================================================
# Compiles all MC simulations into a single HDF5 file with the structure
# needed to train a 3D convolutional neural network.
#
# The dataset combines:
#   - Density maps (network input)
#   - Normalized beam parameters (additional input)
#   - MC dose maps (ground truth / training target)
#   - Uncertainty maps (for weighting loss or filtering noisy voxels)
#
# 10 PARAMETERS — CONSISTENT WITH red_neuronal_EBRT.ipynb:
#   [beam_energy_norm, is_fff, field_x_cm, field_y_cm, ssd_cm,
#    gantry_angle, collimator_angle, patient_type, tumor_depth, tumor_size]
#
# POST-KAGGLE FIXES:
#   1. quality_map → energy_map: separated into beam_energy_norm + is_fff (2 params).
#      Previously combined into a single float (0.0-0.7), now 2 independent
#      floats as the neural network expects.
#   2. Added collimator_angle (was in metadata but not saved).
#   3. Added patient_type ordinal (0=pelvis, 0.5=head_neck, 1.0=lung).
#   4. Removed has_lung (redundant: lung patients always have has_lung=True,
#      the info is already encoded in patient_type).
#   5. Field size extraction fixed: metadata keys are
#      'jaw_size_x_cm' and 'jaw_size_y_cm', not 'size_x_cm' / 'size_y_cm'.
#   6. field_size_x_cm range: 1-40 cm (jaws), field_size_y_cm: 1-22 cm (HD MLC).
#   7. Detailed prints for each processed scenario.
# =============================================================================

import numpy as np
import json
import os
import h5py  # Standard library for accessing HDF5 files


def prepare_ebrt_dataset(batch_dir, output_h5_path, max_scenarios=None):
    """
    Builds the HDF5 training dataset file.

    Reads all successful simulations from batch_dir, generates the
    corresponding analytical density maps, normalizes parameters, and
    packages everything into a single compressed HDF5 file.

    The 10 saved parameters are EXACTLY those expected by the neural
    network (red_neuronal_EBRT.ipynb, Config.n_beam_params = 10):
      [beam_energy_norm, is_fff, field_x_cm, field_y_cm, ssd_cm,
       gantry_angle, collimator_angle, patient_type, tumor_depth, tumor_size]

    Parameters:
        batch_dir: directory with the simulation subdirectories
        output_h5_path: path of the output HDF5 file
        max_scenarios: limit the number of scenarios (useful for debug)

    Returns:
        Dictionary with dataset statistics, or None if it fails
    """
    # === Step 1: Discover available simulations ===
    scenario_dirs = []
    for d in sorted(os.listdir(batch_dir)):
        meta_path = os.path.join(batch_dir, d, f'{d}_metadata.json')
        if os.path.isdir(os.path.join(batch_dir, d)) and os.path.exists(meta_path):
            scenario_dirs.append(d)
    if max_scenarios:
        scenario_dirs = scenario_dirs[:max_scenarios]
    if not scenario_dirs:
        print("  ✗ No simulations found in", batch_dir)
        return None

    print(f"  Building EBRT dataset with {len(scenario_dirs)} simulations")
    print(f"  Directorio: {batch_dir}\n")

    # === Step 2: Get volume shape from the first simulation ===
    first_dose_path = os.path.join(batch_dir, scenario_dirs[0],
                                    f'{scenario_dirs[0]}_dose.mhd')
    if not os.path.exists(first_dose_path):
        print(f"  ✗ Dose file not found: {first_dose_path}")
        return None
    first_dose, first_meta = load_mhd(first_dose_path)
    vol_shape = first_dose.shape  # (Nz, Ny, Nx), typically (200, 200, 200)
    print(f"  Reference shape: {vol_shape} (from {scenario_dirs[0]})")

    # === Step 3: Define parameter normalization ===
    # ──────────────────────────────────────────────────────────────────
    # 10 PARAMETERS — Consistent with red_neuronal_EBRT.ipynb Config:
    #   n_beam_params: int = 10
    #   [beam_energy_norm, is_fff, field_x_cm, field_y_cm, ssd_cm,
    #    gantry_angle, collimator_angle, patient_type, tumor_depth, tumor_size]
    # ──────────────────────────────────────────────────────────────────

    # Beam quality → normalized ordinal energy mapping [0, 1]
    # Consistent with neural network: 0.0=6MV, 0.5=10MV, 1.0=15MV
    energy_map = {
        '6MV':   0.0,   # 6 MV FF
        '6FFF':  0.0,   # 6 MV FFF (same nominal energy, different spectrum)
        '10MV':  0.5,   # 10 MV FF
        '10FFF': 0.5,   # 10 MV FFF
        '15MV':  1.0,   # 15 MV FF
    }

    # Beam quality → is_fff mapping (0 = FF with filter, 1 = FFF without filter)
    fff_map = {
        '6MV':   0.0,
        '6FFF':  1.0,
        '10MV':  0.0,
        '10FFF': 1.0,
        '15MV':  0.0,
    }

    # Patient type → normalized ordinal value mapping [0, 1]
    # Consistent with neural network: 0=pelvis, 0.5=head-neck, 1.0=lung
    patient_type_map = {
        'pelvis':    0.0,
        'head_neck': 0.5,
        'lung':      1.0,
    }

    # Normalization ranges: each parameter is linearly scaled to [0, 1]
    # using the formula: norm = (value - min) / (max - min)
    # CONSISTENT with the normalization table from red_neuronal_EBRT.ipynb
    param_ranges = {
        'beam_energy_norm':    (0.0, 1.0),       # Already encoded as 0-1 ordinal
        'is_fff':              (0.0, 1.0),        # Variable binaria (0=FF, 1=FFF)
        'field_x_cm':          (1.0, 40.0),       # Jaws X: 1-40 cm
        'field_y_cm':          (1.0, 22.0),       # HD MLC Y: maximum 22 cm
        'ssd_cm':              (80.0, 110.0),     # SSD clínico
        'gantry_angle':        (0.0, 360.0),      # Full angle
        'collimator_angle':    (0.0, 360.0),      # Collimator angle
        'patient_type':        (0.0, 1.0),        # Ordinal: pelvis/head_neck/lung
        'tumor_depth':         (2.0, 18.0),       # Depth in cm
        'tumor_size':          (1.0, 8.0),        # Diameter in cm
    }
    n_params = len(param_ranges)  # Must be 10
    n = len(scenario_dirs)

    print(f"  Parameters: {n_params} ({', '.join(param_ranges.keys())})")
    print(f"  Supported qualities: {list(energy_map.keys())}")
    assert n_params == 10, f"ERROR: Expected 10 parameters, got {n_params}"

    # === Step 4: Create HDF5 file with pre-sized datasets ===
    with h5py.File(output_h5_path, 'w') as hf:
        # Density dataset: (N, 1, Dz, Dy, Dx) float32
        # The "1" dimension is the channel (3D-CNN convention)
        # chunks=(1,...) allows reading one scenario at a time without loading all
        # compression='gzip' at level 4 reduces size ~3-5× without being slow
        ds_density = hf.create_dataset('density', shape=(n, 1, *vol_shape),
                                       dtype='float32', chunks=(1, 1, *vol_shape),
                                       maxshape=(None, 1, *vol_shape),
                                       compression='gzip', compression_opts=4)
        # Parameters dataset: (N, 10) float32
        ds_params = hf.create_dataset('params', shape=(n, n_params), dtype='float32',
                                      maxshape=(None, n_params))
        # Dose dataset (ground truth): (N, 1, Dz, Dy, Dx) float32
        ds_dose = hf.create_dataset('dose', shape=(n, 1, *vol_shape),
                                    dtype='float32', chunks=(1, 1, *vol_shape),
                                    maxshape=(None, 1, *vol_shape),
                                    compression='gzip', compression_opts=4)
        # Uncertainty dataset: (N, 1, Dz, Dy, Dx) float32
        ds_uncert = hf.create_dataset('uncertainty', shape=(n, 1, *vol_shape),
                                      dtype='float32', chunks=(1, 1, *vol_shape),
                                      maxshape=(None, 1, *vol_shape),
                                      compression='gzip', compression_opts=4)

        # Global dataset metadata (stored as HDF5 attributes)
        hf.attrs['n_scenarios'] = n
        hf.attrs['volume_shape'] = list(vol_shape)
        hf.attrs['param_names'] = list(param_ranges.keys())
        hf.attrs['param_ranges'] = json.dumps({k: list(v) for k, v in param_ranges.items()})
        hf.attrs['modality'] = 'EBRT_photon'
        hf.attrs['machine'] = 'Varian_TrueBeam_SVC_3.0'

        global_dose_max = 0  # For global normalization at the end
        write_idx = 0        # Write index (may skip simulations with errors)
        skipped = []         # List of skipped scenarios with reason

        # === Step 5: Iterate over each simulation and write to HDF5 ===
        for i, sid in enumerate(scenario_dirs):
            sim_dir = os.path.join(batch_dir, sid)
            with open(os.path.join(sim_dir, f'{sid}_metadata.json')) as f:
                sim_meta = json.load(f)

            # Skip simulations with errors
            if 'error' in sim_meta:
                skipped.append((sid, f"error: {sim_meta['error'][:60]}"))
                print(f"  ✗ {sid}: metadata error — skipping")
                continue

            # Load the dose map
            dose_path = os.path.join(sim_dir, f'{sid}_dose.mhd')
            if not os.path.exists(dose_path):
                skipped.append((sid, "dose file not found"))
                print(f"  ✗ {sid}: dose file not found — skipping")
                continue
            dose_arr, dose_arr_meta = load_mhd(dose_path)

            # Verify that the shape matches the reference
            if dose_arr.shape != vol_shape:
                skipped.append((sid, f"shape {dose_arr.shape} ≠ {vol_shape}"))
                print(f"  ✗ {sid}: incorrect shape {dose_arr.shape}, "
                      f"expected {vol_shape} — skipping")
                continue

            # Build the corresponding analytical density map
            density = build_ebrt_density(sim_meta, vol_shape,
                                         dose_arr_meta['spacing'], dose_arr_meta['offset'])

            # Normalize density ÷ 1.85 (compact bone = 1.0)
            # Consistent with generate_synthetic_phantom() from red_neuronal_EBRT.ipynb
            # which generates already normalized densities: water=0.54, bone=1.0, lung=0.14
            density = density / 1.85

            # --- Extract and normalize the 10 parameters ---
            bq = sim_meta.get('beam', {}).get('beam_quality', '6MV')
            beam_info = sim_meta.get('beam', {})
            geom = sim_meta.get('geometry', {})
            field_meta = sim_meta.get('field', {})

            raw_params = {
                'beam_energy_norm':  energy_map.get(bq, 0.0),
                'is_fff':            fff_map.get(bq, 0.0),
                'field_x_cm':        field_meta.get('jaw_size_x_cm', 10.0),
                'field_y_cm':        field_meta.get('jaw_size_y_cm', 10.0),
                'ssd_cm':            beam_info.get('ssd_cm', 100.0),
                'gantry_angle':      beam_info.get('gantry_angle_deg', 0.0),
                'collimator_angle':  beam_info.get('collimator_angle_deg', 0.0),
                'patient_type':      patient_type_map.get(
                                         sim_meta.get('patient_type', 'pelvis'), 0.0),
                'tumor_depth':       geom.get('tumor_depth_cm', 5.0),
                'tumor_size':        geom.get('tumor_size_cm', 3.0),
            }

            # Normalize each parameter to [0, 1]
            norm_vec = []
            for key in param_ranges:
                lo, hi = param_ranges[key]
                val = raw_params.get(key, lo)
                norm_vec.append((val - lo) / (hi - lo) if hi > lo else 0.0)

            # --- Write to HDF5 ---
            ds_density[write_idx, 0, ...] = density.astype(np.float32)
            ds_params[write_idx, :] = np.array(norm_vec, dtype=np.float32)
            ds_dose[write_idx, 0, ...] = dose_arr.astype(np.float32)

            # Load uncertainty (if exists)
            uncert_path = os.path.join(sim_dir, f'{sid}_dose_uncertainty.mhd')
            if os.path.exists(uncert_path):
                uncert_arr, _ = load_mhd(uncert_path)
                ds_uncert[write_idx, 0, ...] = uncert_arr.astype(np.float32)

            # Update global dose maximum
            if dose_arr.max() > global_dose_max:
                global_dose_max = dose_arr.max()

            ptype = sim_meta.get('patient_type', '?')
            is_fff_str = 'FFF' if raw_params['is_fff'] > 0.5 else 'FF'
            print(f"  ✓ [{write_idx+1}] {sid} | {bq} ({is_fff_str}) | {ptype} | "
                  f"field {raw_params['field_x_cm']}×{raw_params['field_y_cm']} cm² | "
                  f"coll={raw_params['collimator_angle']:.0f}° | "
                  f"dose max={dose_arr.max():.4e} Gy | "
                  f"density [{density.min():.2f}, {density.max():.2f}]")
            write_idx += 1

        # === Step 6: Save final metadata ===
        if write_idx < n:
            # Resize datasets if there were failed simulations
            for dsname in ['density', 'params', 'dose', 'uncertainty']:
                hf[dsname].resize(write_idx, axis=0)

        hf.attrs['n_valid'] = write_idx
        hf.attrs['global_dose_max'] = float(global_dose_max)

    # --- Final summary ---
    file_size_mb = os.path.getsize(output_h5_path) / 1e6
    print(f"\n{'='*60}")
    print(f"  HDF5 dataset created: {output_h5_path}")
    print(f"  Valid scenarios: {write_idx}/{n} "
          f"{'✓ all OK' if write_idx == n else f'({n - write_idx} discarded)'}")
    if skipped:
        for sid, motivo in skipped:
            print(f"    ✗ {sid}: {motivo}")
    print(f"  Volume shape:        {vol_shape}")
    print(f"  Parameters shape: ({write_idx}, {n_params})")
    print(f"  Parameters: {list(param_ranges.keys())}")
    print(f"  Global maximum dose: {global_dose_max:.4e} Gy")
    print(f"  File size:           {file_size_mb:.1f} MB")
    print(f"{'='*60}")

    return {
        'n_valid': write_idx, 'vol_shape': list(vol_shape),
        'n_params': n_params, 'global_dose_max': float(global_dose_max),
    }


# Name consistent with _find_dataset_h5() from red_neuronal_EBRT.ipynb
h5_path = os.path.join(BASE_OUTPUT, 'ebrt_training_dataset.h5')

# --- Execute dataset preparation ---
dataset_stats = prepare_ebrt_dataset(BATCH_OUTPUT, h5_path)

## 13 — Summary and conclusions

### What we have built

This notebook implements a complete Monte Carlo simulation pipeline for external beam radiation therapy (EBRT) **specific to the Varian TrueBeam SVC 3.0 with HD 120 MLC**, the machine at the hospital that will provide us with real data. The pipeline includes:

1. **TrueBeam physical model**: Bremsstrahlung spectra for the 5 beam qualities of the TrueBeam SVC 3.0: **6MV (FF), 6FFF, 10MV (FF), 10FFF, 15MV (FF)**. The dosimetric parameters (dmax, PDD@10cm, D20/D10) come from **real commissioning data of a TrueBeam SVC 3.0** (Saravanakumar et al., 2025).

2. **Complete collimation chain**: Point source at SAD = 100 cm → Y jaws (W, 7.8 cm) → X jaws (W, 7.8 cm) → **HD 120 MLC** (60 pairs: 32 × 2.5 mm central + 28 × 5.0 mm outer, W, 6.5 cm). The individual positions of all 120 leaves conform the field to the tumor contour.

3. **3 patient types**: Pelvis (deep tumors, prominent bone), head-and-neck (compact anatomy, many OARs), lung (extreme heterogeneity, CPE loss). Each type generates realistic anatomical phantoms and is correlated with clinically appropriate beam qualities.

4. **Scenario generation**: Stratified sampling by patient type with patient-energy correlation reflecting real clinical practice. Analytic MLC positions for circular conformation.

5. **Robust execution**: Multiprocessing with timeout, automatic resume, error logging.

6. **Dosimetric analysis**: PDD, lateral profiles, DVH with quality indices (HI, D95, D5), 2D maps with isodose lines.

7. **HDF5 dataset**: Optimized format for 3D convolutional neural networks.

### Fundamental differences from IORT

| Aspect | IORT (electrons) | EBRT (TrueBeam SVC 3.0) |
|---|---|---|
| **Particle** | Direct electrons | Photons (bremsstrahlung) |
| **Energy** | 6-12 MeV (monoenergetic) | Spectrum 0-15 MeV (5 qualities) |
| **Filter** | N/A | FF or FFF (filter-free) |
| **Range** | Finite (R80 ~2-4 cm) | Exponential (infinite) |
| **Surface dose** | ~90-100% | 31-56% (TrueBeam commissioning) |
| **Collimation** | Cone applicator | Jaws + HD 120 MLC (2.5/5.0 mm) |
| **Treatment** | Intraoperative, single field | Fractionated, multiple fields |
| **MC speed** | ~2000 part/s/core | ~150 part/s/core (with MLC) |
| **Patient type** | N/A | Pelvis, head-and-neck, lung |

### Next steps

- **Real DICOM data**: Integrate hospital plans with real MLC positions from Eclipse v17.0.0
- **Validation**: Compare MC PDD and profiles against commissioning data and AcurosXB
- **Patient CT**: Replace analytic phantoms with real CTs (HU → material conversion)
- **VMAT**: Extend to modulated arcs (dynamic MLC positions per control point)
- **3D neural network**: Train with generated dose maps
